# **UNZIP DATA**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
from pathlib import Path
import shutil
import zipfile
import time

ZIP_PATH = Path("/content/drive/MyDrive/IS6002/EXPERIMENT/implementation/data/contracts.zip")

LOCAL_ZIP = Path("/content/contracts.zip")

EXTRACT_DIR = Path("/content/solidity_data")

start = time.time()

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy: {ZIP_PATH}")

print("Đang copy ZIP về Colab...", flush=True)
shutil.copy2(ZIP_PATH, LOCAL_ZIP)

print("Copy thành công!", flush=True)

print("Đang giải nén...", flush=True)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Giải nén thành công!", flush=True)

local_sol_files = list(EXTRACT_DIR.rglob("*.sol"))

print("\n========== KẾT QUẢ ==========")
print(f"Tổng file .sol: {len(local_sol_files):,}")
print(f"Thời gian: {(time.time() - start) / 60:.2f} phút")
print(f"Thư mục giải nén: {EXTRACT_DIR}")

Đang copy ZIP về Colab...
Copy thành công!
Đang giải nén...
Giải nén thành công!

========== KẾT QUẢ ==========
Tổng file .sol: 39,905
Thời gian: 0.29 phút
Thư mục giải nén: /content/solidity_data


# **KIỂM TRA SOLIDITY COMPILER**

In [15]:
%pip install -q py-solc-x

In [16]:

from pathlib import Path
import re
import time
from collections import Counter

EXTRACT_DIR = Path("/content/solidity_data")

if not EXTRACT_DIR.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục: {EXTRACT_DIR}"
    )

sol_files = sorted(EXTRACT_DIR.rglob("*.sol"))

total_files = len(sol_files)

print(f"Tìm thấy {total_files:,} Smart Contract.")

if total_files != 39905:
    raise ValueError(
        f"Số lượng không khớp! Mong đợi 39,905, "
        f"nhưng tìm thấy {total_files:,}."
    )



pragma_counter = Counter()
no_pragma = []
read_errors = []

pattern = re.compile(
    r'\bpragma\s+solidity\s+([^;]+);',
    re.IGNORECASE
)

start_time = time.time()

print(f"\nBắt đầu xử lý {total_files:,} Smart Contract...\n",
      flush=True)


for index, file in enumerate(sol_files, start=1):
    try:
        source = file.read_text(
            encoding="utf-8",
            errors="replace"
        )

        source = re.sub(
            r'/\*.*?\*/|//[^\n]*',
            '',
            source,
            flags=re.DOTALL
        )

        matches = pattern.findall(source)

        if matches:
            for pragma in matches:
                pragma = " ".join(pragma.split())
                pragma_counter[pragma] += 1
        else:
            no_pragma.append(file.name)

    except Exception as e:
        read_errors.append((file.name, str(e)))



    if index % 10000 == 0 or index == total_files:
        elapsed = time.time() - start_time
        avg_time = elapsed / index
        remaining = avg_time * (total_files - index)

        print(
            f"Đã xử lý: {index:,}/{total_files:,} "
            f"({index / total_files * 100:.1f}%)"
        )

        print(
            f"Thời gian đã chạy: {elapsed / 60:.2f} phút"
        )

        print(
            f"Thời gian còn lại (ước tính): "
            f"{remaining / 60:.2f} phút"
        )

        print("-" * 50, flush=True)



print("\n========== THỐNG KÊ PRAGMA SOLIDITY ==========")

print("Tổng file .sol:", total_files)

print(
    "Số khai báo pragma tìm thấy:",
    sum(pragma_counter.values())
)

print("Số file không có pragma:", len(no_pragma))
print("Số file đọc lỗi:", len(read_errors))

print("\n========== TOP 30 PRAGMA PHỔ BIẾN ==========")

for pragma, count in pragma_counter.most_common(30):
    print(f"{pragma:<30} : {count:,}")

if no_pragma:
    print("\nVí dụ file không có pragma:")
    print(no_pragma[:10])

if read_errors:
    print("\nVí dụ file đọc lỗi:")
    print(read_errors[:5])

print(
    f"\nTổng thời gian xử lý: "
    f"{(time.time() - start_time) / 60:.2f} phút"
)

Tìm thấy 39,905 Smart Contract.

Bắt đầu xử lý 39,905 Smart Contract...

Đã xử lý: 10,000/39,905 (25.1%)
Thời gian đã chạy: 0.10 phút
Thời gian còn lại (ước tính): 0.31 phút
--------------------------------------------------
Đã xử lý: 20,000/39,905 (50.1%)
Thời gian đã chạy: 0.18 phút
Thời gian còn lại (ước tính): 0.18 phút
--------------------------------------------------
Đã xử lý: 30,000/39,905 (75.2%)
Thời gian đã chạy: 0.25 phút
Thời gian còn lại (ước tính): 0.08 phút
--------------------------------------------------
Đã xử lý: 39,905/39,905 (100.0%)
Thời gian đã chạy: 0.34 phút
Thời gian còn lại (ước tính): 0.00 phút
--------------------------------------------------

========== THỐNG KÊ PRAGMA SOLIDITY ==========
Tổng file .sol: 39905
Số khai báo pragma tìm thấy: 45793
Số file không có pragma: 1059
Số file đọc lỗi: 0

========== TOP 30 PRAGMA PHỔ BIẾN ==========
^0.4.24                        : 7,595
^0.4.18                        : 6,718
^0.4.25                        : 2,765
^

In [17]:
import pandas as pd

# Chuyển kết quả thống kê thành DataFrame
df_pragma = pd.DataFrame(
    pragma_counter.items(),
    columns=["pragma", "count"]
)

# Sắp xếp theo số lượng giảm dần
df_pragma = df_pragma.sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

print("========== TỔNG HỢP ==========")
print("Số kiểu khai báo pragma khác nhau:", len(df_pragma))
print("Tổng số khai báo:", df_pragma["count"].sum())

print("\n========== TOÀN BỘ PRAGMA ==========")
display(df_pragma)

========== TỔNG HỢP ==========
Số kiểu khai báo pragma khác nhau: 183
Tổng số khai báo: 45793

========== TOÀN BỘ PRAGMA ==========


,pragma,count
0,^0.4.24,7595
1,^0.4.18,6718
2,^0.4.25,2765
3,^0.4.16,2608
4,^0.4.23,2293
...,...,...
178,>=0.4.26 <0.5.0,1
179,>0.4.23,1
180,^ 0.5.1,1
181,>=0.4.1,1


In [18]:
version_counter = Counter()
unrecognized = []

for pragma, count in pragma_counter.items():

    versions = re.findall(
        r'(\d+)\.(\d+)\.(\d+)',
        pragma
    )

    if versions:
        branches = {
            f"{major}.{minor}.x"
            for major, minor, patch in versions
        }

        for branch in branches:
            version_counter[branch] += count
    else:
        unrecognized.append((pragma, count))

print("========== PHÂN BỐ NHÁNH SOLIDITY ==========")

for branch, count in sorted(version_counter.items()):
    print(f"{branch:<12}: {count:,}")

print("\n========== KHÔNG NHẬN DIỆN ==========")

print("Số kiểu pragma:", len(unrecognized))

for pragma, count in unrecognized[:20]:
    print(f"{pragma}: {count}")

========== PHÂN BỐ NHÁNH SOLIDITY ==========
0.4.x       : 39,968
0.5.x       : 5,590
0.6.x       : 648
0.7.x       : 8
6.0.x       : 1

========== KHÔNG NHẬN DIỆN ==========
Số kiểu pragma: 29
^ 0.4 .2: 13
^ 0.5 .1: 1
^0.4: 4
^ 0.4 .18: 6
^ 0.4 .9: 127
^ 0.4 .24: 10
^0.5: 13
^ 0.4 .11: 10
^ 0.4 .19: 8
^ 0.4 .6: 5
0.4 .21: 1
^ 0.4 .16: 3
^ 0.4 .0: 4
^ 0.4 .21: 2
^ 0.4 .25: 2
0.4 .19: 5
^ 0.4 .17: 4
0.4 .25: 7
^ 0.4 .23: 2
< 0.6: 1


In [19]:
df_unrecognized = pd.DataFrame(
    unrecognized,
    columns=["pragma", "count"]
)

df_unrecognized = df_unrecognized.sort_values(
    "count",
    ascending=False
).reset_index(drop=True)

print("========== PRAGMA CẦN KIỂM TRA ==========")

print("Số kiểu pragma:", len(df_unrecognized))
print("Tổng số lượt xuất hiện:", df_unrecognized["count"].sum())

display(df_unrecognized)

========== PRAGMA CẦN KIỂM TRA ==========
Số kiểu pragma: 29
Tổng số lượt xuất hiện: 245


,pragma,count
0,^ 0.4 .9,127
1,^ 0.4 .2,13
2,^0.5,13
3,^ 0.4 .24,10
4,^ 0.4 .11,10
5,^ 0.4 .19,8
6,0.4 .25,7
7,^ 0.4 .8,6
8,^ 0.4 .18,6
9,^ 0.4 .6,5


In [20]:
from collections import defaultdict

target_pragmas = {
    "^ 0.4 .9",
    "^0.5",
    "<= 0.6",
    ">=0.5",
    "^0.4. 21"
}

examples = defaultdict(list)
multi_pragma_files = []

for file in sol_files:
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    cleaned = re.sub(
        r'/\*.*?\*/|//[^\n]*',
        '',
        source,
        flags=re.DOTALL
    )

    pragmas = [
        " ".join(p.split())
        for p in pattern.findall(cleaned)
    ]

    if len(pragmas) > 1:
        multi_pragma_files.append(
            (file.name, pragmas)
        )

    for pragma in pragmas:
        if (
            pragma in target_pragmas
            and len(examples[pragma]) < 3
        ):
            examples[pragma].append(file.name)

print("========== PRAGMA BẤT THƯỜNG ==========")

for pragma in sorted(target_pragmas):
    print(f"\nPragma: {pragma}")
    print("Ví dụ file:", examples[pragma])

print("\n========== FILE CÓ NHIỀU PRAGMA ==========")

print("Số file:", len(multi_pragma_files))

for filename, pragmas in multi_pragma_files[:10]:
    print(f"\n{filename}")
    print("Pragma:", pragmas)

========== PRAGMA BẤT THƯỜNG ==========

Pragma: <= 0.6
Ví dụ file: ['C031584.sol', 'C034608.sol']

Pragma: >=0.5
Ví dụ file: ['C029068.sol']

Pragma: ^ 0.4 .9
Ví dụ file: ['C000868.sol', 'C000924.sol', 'C001156.sol']

Pragma: ^0.4. 21
Ví dụ file: ['C019139.sol', 'C031560.sol']

Pragma: ^0.5
Ví dụ file: ['C001586.sol', 'C003340.sol', 'C009097.sol']

========== FILE CÓ NHIỀU PRAGMA ==========
Số file: 1334

C000016.sol
Pragma: ['^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.0']

C000017.sol
Pragma: ['^0.5.0', '^0.5.0']

C000066.sol
Pragma: ['^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.0']

C000067.sol
Pragma: ['^0.5.0', '^0.5.0']

C000116.sol
Pragma: ['^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.0']

C000117.sol
Pragma: ['^0.5.0', '^0.5.0']

C000166.sol
Pragma: ['^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.0']

C000167.sol
Pragma: ['^0.5.0', '^0.5.0']

C000216.sol
Pragma: ['^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.2', '^0.5.0']

C000217.sol


In [21]:
files_with_import = []
files_without_import = []
import_counter = Counter()

import_pattern = re.compile(
    r'\bimport\s+(?:[^;]*?\bfrom\s+)?["\']([^"\']+)["\']\s*;',
    re.IGNORECASE
)

for file in sol_files:
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    cleaned = re.sub(
        r'/\*.*?\*/|//[^\n]*',
        '',
        source,
        flags=re.DOTALL
    )

    imports = import_pattern.findall(cleaned)

    if imports:
        files_with_import.append((file.name, imports))

        for imp in imports:
            import_counter[imp] += 1
    else:
        files_without_import.append(file.name)

print("========== THỐNG KÊ IMPORT ==========")
print("Tổng file:", len(sol_files))
print("File có import:", len(files_with_import))
print("File không có import:", len(files_without_import))

print("\n========== TOP 20 IMPORT ==========")

for imp, count in import_counter.most_common(20):
    print(f"{count:>6,} | {imp}")

print("\n========== VÍ DỤ FILE CÓ IMPORT ==========")

for filename, imports in files_with_import[:10]:
    print(f"{filename}: {imports}")

========== THỐNG KÊ IMPORT ==========
Tổng file: 39905
File có import: 0
File không có import: 39905

========== TOP 20 IMPORT ==========

========== VÍ DỤ FILE CÓ IMPORT ==========


In [22]:
import solcx

available_versions = solcx.get_installable_solc_versions()

installed_versions = solcx.get_installed_solc_versions()

print("========== COMPILER CÓ THỂ CÀI ==========")

for version in available_versions:
    if version.major == 0 and version.minor in [4, 5, 6, 7]:
        print(version)

print("\n========== COMPILER ĐÃ CÀI ==========")
print(installed_versions)

========== COMPILER CÓ THỂ CÀI ==========
0.7.6
0.7.5
0.7.4
0.7.3
0.7.2
0.7.1
0.7.0
0.6.12
0.6.11
0.6.10
0.6.9
0.6.8
0.6.7
0.6.6
0.6.5
0.6.4
0.6.3
0.6.2
0.6.1
0.6.0
0.5.17
0.5.16
0.5.15
0.5.14
0.5.13
0.5.12
0.5.11
0.5.10
0.5.9
0.5.8
0.5.7
0.5.6
0.5.5
0.5.4
0.5.3
0.5.2
0.5.1
0.5.0
0.4.26
0.4.25
0.4.24
0.4.23
0.4.22
0.4.21
0.4.20
0.4.19
0.4.18
0.4.17
0.4.16
0.4.15
0.4.14
0.4.13
0.4.12
0.4.11

========== COMPILER ĐÃ CÀI ==========
[]


In [23]:
# Bộ compiler ban đầu
versions_to_install = [
    "0.4.18",
    "0.4.19",
    "0.4.21",
    "0.4.24",
    "0.4.25",
    "0.4.26",
    "0.5.17",
    "0.6.12",
    "0.7.6"
]

start_time = time.time()

for i, version in enumerate(versions_to_install, start=1):
    print(
        f"[{i}/{len(versions_to_install)}] "
        f"Đang cài solc {version}...",
        flush=True
    )

    try:
        solcx.install_solc(version)
        print(f"  Thành công: {version}", flush=True)

    except Exception as e:
        print(f"  Thất bại: {version}")
        print(f"  Lỗi: {e}")

print("\n========== COMPILER ĐÃ CÀI ==========")

installed = solcx.get_installed_solc_versions()

for version in installed:
    print(version)

print(
    f"\nTổng thời gian: "
    f"{(time.time() - start_time) / 60:.2f} phút"
)

[1/9] Đang cài solc 0.4.18...
  Thành công: 0.4.18
[2/9] Đang cài solc 0.4.19...
  Thành công: 0.4.19
[3/9] Đang cài solc 0.4.21...
  Thành công: 0.4.21
[4/9] Đang cài solc 0.4.24...
  Thành công: 0.4.24
[5/9] Đang cài solc 0.4.25...
  Thành công: 0.4.25
[6/9] Đang cài solc 0.4.26...
  Thành công: 0.4.26
[7/9] Đang cài solc 0.5.17...
  Thành công: 0.5.17
[8/9] Đang cài solc 0.6.12...
  Thành công: 0.6.12
[9/9] Đang cài solc 0.7.6...
  Thành công: 0.7.6

========== COMPILER ĐÃ CÀI ==========
0.7.6
0.6.12
0.5.17
0.4.26
0.4.25
0.4.24
0.4.21
0.4.19
0.4.18

Tổng thời gian: 0.03 phút


In [25]:
# Các nhóm cần thử nghiệm
test_groups = {
    "^0.4.18": "0.4.26",
    "0.4.24": "0.4.24",
    "0.4.25": "0.4.25",
    "^0.5.0": "0.5.17",
    "^0.5.2": "0.5.17"
}

MAX_PER_GROUP = 10

selected = defaultdict(list)

# Chọn file cho từng nhóm
for file in sol_files:
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    cleaned = re.sub(
        r'/\*.*?\*/|//[^\n]*',
        '',
        source,
        flags=re.DOTALL
    )

    pragmas = [
        " ".join(p.split())
        for p in pattern.findall(cleaned)
    ]

    # Chỉ chọn file có các pragma giống nhau
    if pragmas and len(set(pragmas)) == 1:
        pragma = pragmas[0]

        if (
            pragma in test_groups
            and len(selected[pragma]) < MAX_PER_GROUP
        ):
            selected[pragma].append(file)

results = []
start_time = time.time()

for pragma, compiler in test_groups.items():
    files = selected[pragma]

    print(
        f"\nĐang kiểm tra {pragma} "
        f"với solc {compiler}: {len(files)} file",
        flush=True
    )

    for file in files:
        try:
            source = file.read_text(
                encoding="utf-8",
                errors="replace"
            )

            compiled = solcx.compile_source(
                source,
                output_values=["bin", "bin-runtime"],
                solc_version=compiler
            )

            has_bytecode = any(
                artifact.get("bin")
                for artifact in compiled.values()
            )

            status = (
                "SUCCESS"
                if has_bytecode
                else "NO_BYTECODE"
            )

            error = ""

        except Exception as e:
            status = "FAILED"
            error = str(e)[:500]

        results.append({
            "contract_id": file.stem,
            "pragma": pragma,
            "compiler": compiler,
            "status": status,
            "error": error
        })

df_multi_test = pd.DataFrame(results)

print("\n========== KẾT QUẢ THEO NHÓM ==========")

summary = pd.crosstab(
    [df_multi_test["pragma"], df_multi_test["compiler"]],
    df_multi_test["status"]
)

display(summary)

print("\n========== LỖI BIÊN DỊCH ==========")

display(
    df_multi_test[
        df_multi_test["status"] == "FAILED"
    ][["contract_id", "pragma", "compiler", "error"]]
)

print(
    f"\nTổng thời gian: "
    f"{time.time() - start_time:.2f} giây"
)


Đang kiểm tra ^0.4.18 với solc 0.4.26: 10 file

Đang kiểm tra 0.4.24 với solc 0.4.24: 10 file

Đang kiểm tra 0.4.25 với solc 0.4.25: 10 file

Đang kiểm tra ^0.5.0 với solc 0.5.17: 10 file

Đang kiểm tra ^0.5.2 với solc 0.5.17: 10 file

========== KẾT QUẢ THEO NHÓM ==========


,status,SUCCESS
pragma,compiler,
0.4.24,0.4.24,10
0.4.25,0.4.25,10
^0.4.18,0.4.26,10
^0.5.0,0.5.17,10
^0.5.2,0.5.17,10



========== LỖI BIÊN DỊCH ==========


,contract_id,pragma,compiler,error



Tổng thời gian: 1.72 giây


In [26]:
from packaging.version import Version

installed_versions = sorted(
    [str(v) for v in solcx.get_installed_solc_versions()],
    key=Version,
    reverse=True
)

def normalize_pragma(pragma):
    pragma = re.sub(r'\s*\.\s*', '.', pragma)
    pragma = re.sub(r'([<>=^~])\s+', r'\1', pragma)
    return " ".join(pragma.split())

# Kiểm tra một compiler có thỏa mãn pragma không
def satisfies_pragma(pragma, compiler):
    pragma = normalize_pragma(pragma)
    v = Version(compiler)

    constraints = re.findall(
        r'(\^|~|>=|<=|>|<|=)?\s*(\d+(?:\.\d+){0,2})',
        pragma
    )

    if not constraints:
        return False

    for operator, version_text in constraints:
        parts = [int(x) for x in version_text.split(".")]
        specified_parts = len(parts)
        parts += [0] * (3 - len(parts))

        target = Version(".".join(map(str, parts)))

        if operator == "^":
            if parts[0] != 0:
                upper = Version(f"{parts[0] + 1}.0.0")
            elif parts[1] != 0:
                upper = Version(f"0.{parts[1] + 1}.0")
            else:
                upper = Version(f"0.0.{parts[2] + 1}")

            valid = target <= v < upper

        elif operator == "~":
            upper = Version(f"{parts[0]}.{parts[1] + 1}.0")
            valid = target <= v < upper

        elif operator == ">=":
            valid = v >= target

        elif operator == "<=":
            valid = v <= target

        elif operator == ">":
            valid = v > target

        elif operator == "<":
            valid = v < target

        elif operator in ("", "="):
            if specified_parts == 3:
                valid = v == target
            elif specified_parts == 2:
                valid = (
                    v.major == parts[0]
                    and v.minor == parts[1]
                )
            else:
                valid = v.major == parts[0]

        else:
            valid = False

        if not valid:
            return False

    return True


# Phân tích toàn bộ 39.905 file
compiler_mapping = []
pragma_pattern = re.compile(
    r'\bpragma\s+solidity\s+([^;]+);',
    re.IGNORECASE
)

for file in sol_files:
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    cleaned = re.sub(
        r'/\*.*?\*/|//[^\n]*',
        '',
        source,
        flags=re.DOTALL
    )

    pragmas = [
        normalize_pragma(p)
        for p in pragma_pattern.findall(cleaned)
    ]

    compatible = [
        version
        for version in installed_versions
        if pragmas and all(
            satisfies_pragma(p, version)
            for p in pragmas
        )
    ]

    if not pragmas:
        status = "NO_PRAGMA"
    elif compatible:
        status = "COMPILER_AVAILABLE"
    else:
        status = "NO_COMPATIBLE_COMPILER"

    compiler_mapping.append({
        "contract_id": file.stem,
        "pragma_count": len(pragmas),
        "pragmas": " | ".join(pragmas),
        "selected_compiler": compatible[0] if compatible else None,
        "status": status
    })

df_compiler_mapping = pd.DataFrame(compiler_mapping)

print("========== KẾT QUẢ PHÂN TÍCH ==========")
print(df_compiler_mapping["status"].value_counts())

print("\n========== PHÂN BỐ COMPILER ĐƯỢC CHỌN ==========")
print(
    df_compiler_mapping["selected_compiler"]
    .value_counts()
)

print("\n========== FILE CHƯA CÓ COMPILER PHÙ HỢP ==========")
display(
    df_compiler_mapping[
        df_compiler_mapping["status"] == "NO_COMPATIBLE_COMPILER"
    ].head(20)
)

========== KẾT QUẢ PHÂN TÍCH ==========
status
COMPILER_AVAILABLE        38325
NO_PRAGMA                  1059
NO_COMPATIBLE_COMPILER      521
Name: count, dtype: int64

========== PHÂN BỐ COMPILER ĐƯỢC CHỌN ==========
selected_compiler
0.4.26    33109
0.5.17     2703
0.4.24     1180
0.4.25      599
0.4.19      317
0.4.21      237
0.7.6       172
0.6.12        8
Name: count, dtype: int64

========== FILE CHƯA CÓ COMPILER PHÙ HỢP ==========


,contract_id,pragma_count,pragmas,selected_compiler,status
358,C000359,1,0.4.9,None,NO_COMPATIBLE_COMPILER
494,C000497,12,0.5.0 | 0.5.0 | 0.5.0 | 0.5.0 | 0.5.0 | 0.5.0 ...,None,NO_COMPATIBLE_COMPILER
495,C000498,1,0.5.6,None,NO_COMPATIBLE_COMPILER
497,C000500,1,0.5.8,None,NO_COMPATIBLE_COMPILER
506,C000509,1,0.5.6,None,NO_COMPATIBLE_COMPILER
509,C000512,1,0.5.8,None,NO_COMPATIBLE_COMPILER
547,C000551,1,0.4.17,None,NO_COMPATIBLE_COMPILER
550,C000554,1,0.5.2,None,NO_COMPATIBLE_COMPILER
577,C000582,1,0.4.14,None,NO_COMPATIBLE_COMPILER
582,C000587,1,0.5.8,None,NO_COMPATIBLE_COMPILER


In [27]:
# Lấy 521 file chưa có compiler
df_missing = df_compiler_mapping[
    df_compiler_mapping["status"] == "NO_COMPATIBLE_COMPILER"
].copy()

# Thống kê toàn bộ pragma trong nhóm này
missing_pragma_counter = Counter()

for pragmas_text in df_missing["pragmas"]:
    pragmas = pragmas_text.split(" | ")

    for pragma in pragmas:
        missing_pragma_counter[pragma] += 1

print("========== TOP 40 PRAGMA CHƯA CÓ COMPILER ==========")

for pragma, count in missing_pragma_counter.most_common(40):
    print(f"{pragma:<35} : {count:,}")

# Xác định phiên bản cố định
exact_version_pattern = re.compile(r"^\d+\.\d+\.\d+$")

exact_versions = Counter()

for pragma, count in missing_pragma_counter.items():
    if exact_version_pattern.fullmatch(pragma):
        exact_versions[pragma] += count

print("\n========== PHIÊN BẢN CỐ ĐỊNH CẦN KIỂM TRA ==========")

for version, count in sorted(exact_versions.items()):
    print(f"solc {version:<10} : {count:,} khai báo")

print("\n========== TỔNG KẾT ==========")
print("Số file chưa có compiler:", len(df_missing))
print("Số kiểu pragma:", len(missing_pragma_counter))
print("Số phiên bản cố định khác nhau:", len(exact_versions))

========== TOP 40 PRAGMA CHƯA CÓ COMPILER ==========
0.5.7                               : 136
0.5.0                               : 128
^0.5.2                              : 121
0.5.8                               : 115
0.4.11                              : 107
^0.5.0                              : 105
0.5.2                               : 82
0.5.6                               : 47
0.4.8                               : 27
0.4.17                              : 24
0.4.4                               : 12
^0.5.7                              : 12
0.4.10                              : 10
0.4.14                              : 9
0.4.9                               : 2
>0.4.13                             : 2
0.4.6                               : 2
>=0.5.0 <0.6.0                      : 1
^0.5.6                              : 1

========== PHIÊN BẢN CỐ ĐỊNH CẦN KIỂM TRA ==========
solc 0.4.10     : 10 khai báo
solc 0.4.11     : 107 khai báo
solc 0.4.14     : 9 khai báo
solc 0.4.17     : 24 kha

In [28]:
versions_to_install = [
    "0.4.11",
    "0.4.14",
    "0.4.17",
    "0.5.0",
    "0.5.2",
    "0.5.6",
    "0.5.7",
    "0.5.8"
]

available = {
    str(v) for v in solcx.get_installable_solc_versions()
}

start_time = time.time()

for i, version in enumerate(versions_to_install, start=1):

    print(
        f"[{i}/{len(versions_to_install)}] "
        f"Đang cài solc {version}...",
        flush=True
    )

    if version not in available:
        print(f"  Không có trong danh sách tải: {version}")
        continue

    try:
        solcx.install_solc(version)
        print(f"  Thành công: {version}", flush=True)

    except Exception as e:
        print(f"  Thất bại: {version}")
        print(f"  Lỗi: {e}")

print("\n========== COMPILER ĐÃ CÀI ==========")

installed = solcx.get_installed_solc_versions()

for version in installed:
    print(version)

print("\nTổng số compiler:", len(installed))

print(
    f"Tổng thời gian: "
    f"{(time.time() - start_time):.2f} giây"
)

[1/8] Đang cài solc 0.4.11...
  Thành công: 0.4.11
[2/8] Đang cài solc 0.4.14...
  Thành công: 0.4.14
[3/8] Đang cài solc 0.4.17...
  Thành công: 0.4.17
[4/8] Đang cài solc 0.5.0...
  Thành công: 0.5.0
[5/8] Đang cài solc 0.5.2...
  Thành công: 0.5.2
[6/8] Đang cài solc 0.5.6...
  Thành công: 0.5.6
[7/8] Đang cài solc 0.5.7...
  Thành công: 0.5.7
[8/8] Đang cài solc 0.5.8...
  Thành công: 0.5.8

========== COMPILER ĐÃ CÀI ==========
0.7.6
0.6.12
0.5.17
0.5.8
0.5.7
0.5.6
0.5.2
0.5.0
0.4.26
0.4.25
0.4.24
0.4.21
0.4.19
0.4.18
0.4.17
0.4.14
0.4.11

Tổng số compiler: 17
Tổng thời gian: 1.64 giây


In [29]:
installed_versions = sorted(
    [str(v) for v in solcx.get_installed_solc_versions()],
    key=Version,
    reverse=True
)

print("Tổng số compiler đã cài:", len(installed_versions))

# Cập nhật danh sách compiler tương thích
def update_compiler_mapping(row):

    if row["status"] == "NO_PRAGMA":
        return pd.Series({
            "selected_compiler": None,
            "status": "NO_PRAGMA"
        })

    pragmas = row["pragmas"].split(" | ")

    compatible = [
        version
        for version in installed_versions
        if all(
            satisfies_pragma(p, version)
            for p in pragmas
        )
    ]

    return pd.Series({
        "selected_compiler": compatible[0] if compatible else None,
        "status": (
            "COMPILER_AVAILABLE"
            if compatible
            else "NO_COMPATIBLE_COMPILER"
        )
    })

df_compiler_mapping[
    ["selected_compiler", "status"]
] = df_compiler_mapping.apply(
    update_compiler_mapping,
    axis=1
)

print("\n========== KẾT QUẢ SAU KHI CÀI 17 COMPILER ==========")

print(df_compiler_mapping["status"].value_counts())

print("\n========== PHÂN BỐ COMPILER ==========")

print(
    df_compiler_mapping["selected_compiler"]
    .value_counts()
)

print("\n========== FILE CHƯA CÓ COMPILER ==========")

remaining = df_compiler_mapping[
    df_compiler_mapping["status"] == "NO_COMPATIBLE_COMPILER"
]

print("Số file còn lại:", len(remaining))

display(
    remaining[
        ["contract_id", "pragmas", "status"]
    ].head(30)
)

Tổng số compiler đã cài: 17

========== KẾT QUẢ SAU KHI CÀI 17 COMPILER ==========
status
COMPILER_AVAILABLE        38793
NO_PRAGMA                  1059
NO_COMPATIBLE_COMPILER       53
Name: count, dtype: int64

========== PHÂN BỐ COMPILER ==========
selected_compiler
0.4.26    33109
0.5.17     2703
0.4.24     1180
0.4.25      599
0.4.19      317
0.4.21      237
0.7.6       172
0.4.11      106
0.5.7        86
0.5.2        81
0.5.0        66
0.5.8        61
0.5.6        44
0.4.17       15
0.4.14        9
0.6.12        8
Name: count, dtype: int64

========== FILE CHƯA CÓ COMPILER ==========
Số file còn lại: 53


,contract_id,pragmas,status
358,C000359,0.4.9,NO_COMPATIBLE_COMPILER
799,C000823,0.4.10,NO_COMPATIBLE_COMPILER
2107,C002244,0.4.8,NO_COMPATIBLE_COMPILER
2205,C002346,0.4.8,NO_COMPATIBLE_COMPILER
3729,C004021,0.4.4,NO_COMPATIBLE_COMPILER
4962,C005356,0.4.10,NO_COMPATIBLE_COMPILER
5108,C005508,0.4.8,NO_COMPATIBLE_COMPILER
5655,C006113,0.4.4,NO_COMPATIBLE_COMPILER
5779,C006248,0.4.8,NO_COMPATIBLE_COMPILER
6664,C007219,0.4.8,NO_COMPATIBLE_COMPILER


In [30]:
# Lọc những file chưa có compiler tương thích
df_remaining = df_compiler_mapping[
    df_compiler_mapping["status"] == "NO_COMPATIBLE_COMPILER"
].copy()

print("========== THỐNG KÊ 53 FILE ==========")

print("Tổng số file:", len(df_remaining))

# Thống kê theo toàn bộ biểu thức pragma của từng file
pragma_counts = Counter(df_remaining["pragmas"])

for pragma, count in pragma_counts.most_common():
    print(f"{pragma:<35} : {count:,} file")

print("\n========== KIỂM TRA TÍNH NHẤT QUÁN ==========")

print("Tổng số file được thống kê:", sum(pragma_counts.values()))

print("\n========== DANH SÁCH CONTRACT ID ==========")

display(
    df_remaining[
        ["contract_id", "pragmas"]
    ].reset_index(drop=True)
)

========== THỐNG KÊ 53 FILE ==========
Tổng số file: 53
0.4.8                               : 27 file
0.4.4                               : 12 file
0.4.10                              : 10 file
0.4.9                               : 2 file
0.4.6                               : 2 file

========== KIỂM TRA TÍNH NHẤT QUÁN ==========
Tổng số file được thống kê: 53

========== DANH SÁCH CONTRACT ID ==========


,contract_id,pragmas
0,C000359,0.4.9
1,C000823,0.4.10
2,C002244,0.4.8
3,C002346,0.4.8
4,C004021,0.4.4
5,C005356,0.4.10
6,C005508,0.4.8
7,C006113,0.4.4
8,C006248,0.4.8
9,C007219,0.4.8


In [31]:
old_versions = [
    "0.4.4",
    "0.4.6",
    "0.4.8",
    "0.4.9",
    "0.4.10"
]

install_results = {}

start_time = time.time()

for index, version in enumerate(old_versions, start=1):

    print(
        f"\n[{index}/{len(old_versions)}] "
        f"Đang thử cài solc {version}...",
        flush=True
    )

    try:
        solcx.install_solc(version)

        install_results[version] = "SUCCESS"

        print(f"Thành công: {version}")

    except Exception as e:

        install_results[version] = "FAILED"

        print(f"Không cài được: {version}")
        print(f"Nguyên nhân: {str(e)[:500]}")

print("\n========== KẾT QUẢ ==========")

for version, status in install_results.items():
    print(f"solc {version:<8}: {status}")

print("\n========== COMPILER ĐÃ CÀI ==========")

installed = solcx.get_installed_solc_versions()

for version in installed:
    print(version)

print("\nTổng compiler:", len(installed))

print(
    f"Thời gian: "
    f"{time.time() - start_time:.2f} giây"
)


[1/5] Đang thử cài solc 0.4.4...
Không cài được: 0.4.4
Nguyên nhân: py-solc-x does not support solc versions <0.4.11

[2/5] Đang thử cài solc 0.4.6...
Không cài được: 0.4.6
Nguyên nhân: py-solc-x does not support solc versions <0.4.11

[3/5] Đang thử cài solc 0.4.8...
Không cài được: 0.4.8
Nguyên nhân: py-solc-x does not support solc versions <0.4.11

[4/5] Đang thử cài solc 0.4.9...
Không cài được: 0.4.9
Nguyên nhân: py-solc-x does not support solc versions <0.4.11

[5/5] Đang thử cài solc 0.4.10...
Không cài được: 0.4.10
Nguyên nhân: py-solc-x does not support solc versions <0.4.11

========== KẾT QUẢ ==========
solc 0.4.4   : FAILED
solc 0.4.6   : FAILED
solc 0.4.8   : FAILED
solc 0.4.9   : FAILED
solc 0.4.10  : FAILED

========== COMPILER ĐÃ CÀI ==========
0.7.6
0.6.12
0.5.17
0.5.8
0.5.7
0.5.6
0.5.2
0.5.0
0.4.26
0.4.25
0.4.24
0.4.21
0.4.19
0.4.18
0.4.17
0.4.14
0.4.11

Tổng compiler: 17
Thời gian: 0.01 giây


In [32]:
import subprocess

commands = [
    ["node", "--version"],
    ["npm", "--version"]
]

for command in commands:
    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=30
        )

        print("Lệnh:", " ".join(command))
        print("Kết quả:", result.stdout.strip())

        if result.returncode != 0:
            print("Lỗi:", result.stderr.strip())

    except Exception as e:
        print("Lỗi:", str(e))

Lệnh: node --version
Kết quả: v22.23.4
Lệnh: npm --version
Kết quả: 10.8.2


In [33]:
!mkdir -p /content/solc_legacy
!cd /content/solc_legacy && npm init -y > /dev/null
!cd /content/solc_legacy && npm install solc --no-audit --no-fund

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 8 packages in 1s
⠹

In [34]:
!cd /content/solc_legacy && node -e "const solc = require('solc'); console.log('solc-js:', solc.version())"

solc-js: 0.8.38+commit.1835d247.Emscripten.clang


In [36]:
LEGACY_DIR = Path("/content/solc_legacy")
LEGACY_DIR.mkdir(parents=True, exist_ok=True)

script = r"""
const solc = require('solc');

solc.loadRemoteVersion('v0.4.8+commit.60cc1668', (error, compiler) => {
    if (error) {
        console.error('DOWNLOAD_FAILED:', error.message);
        process.exit(1);
    }

    console.log('COMPILER_LOADED:', compiler.version());
});
"""

script_path = LEGACY_DIR / "test_legacy.js"
script_path.write_text(script, encoding="utf-8")

result = subprocess.run(
    ["node", str(script_path)],
    cwd=str(LEGACY_DIR),
    capture_output=True,
    text=True,
    timeout=120
)

print("========== STDOUT ==========")
print(result.stdout)

print("========== STDERR ==========")
print(result.stderr)

print("Exit code:", result.returncode)

========== STDOUT ==========
COMPILER_LOADED: 0.4.8+commit.60cc1668.Emscripten.clang

========== STDERR ==========
(node:53191) V8: soljson-v0.4.8+commit.60cc1668.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)

Exit code: 0


In [39]:
LEGACY_DIR = Path("/content/solc_legacy")
CONTRACT_FILE = Path("/content/solidity_data/C002244.sol")

# Kiểm tra file Solidity
if not CONTRACT_FILE.exists():
    matches = list(
        Path("/content/solidity_data").rglob("C002244.sol")
    )
    if matches:
        CONTRACT_FILE = matches[0]
    else:
        raise FileNotFoundError(
            "Không tìm thấy C002244.sol. "
            "Kiểm tra lại dữ liệu đã giải nén."
        )

# Script biên dịch bằng solc-js 0.4.8
script = r"""
const solc = require('solc');
const fs = require('fs');

const filePath = process.argv[2];
const source = fs.readFileSync(filePath, 'utf8');

solc.loadRemoteVersion(
    'v0.4.8+commit.60cc1668',
    (error, compiler) => {
        if (error) {
            console.error('DOWNLOAD_FAILED:', error.message);
            process.exit(1);
        }

        try {
            const output = compiler.compile(source, 0);

            if (output.errors) {
                console.log('========== COMPILER MESSAGES ==========');
                console.log(output.errors.join('\n'));
            }

            const contracts = output.contracts || {};
            const names = Object.keys(contracts);

            console.log('COMPILER:', compiler.version());
            console.log('CONTRACTS_FOUND:', names.length);

            for (const name of names) {
                const artifact = contracts[name];

                const bytecode = artifact.bytecode || '';
                const runtime = artifact.runtimeBytecode || '';

                console.log('CONTRACT:', name);
                console.log('CREATION_BYTECODE_LENGTH:', bytecode.length / 2);
                console.log('RUNTIME_BYTECODE_LENGTH:', runtime.length / 2);
            }

            if (names.length === 0) {
                process.exitCode = 2;
            }

        } catch (e) {
            console.error('COMPILE_FAILED:', e.message);
            process.exitCode = 1;
        }
    }
);
"""

script_path = LEGACY_DIR / "compile_legacy_test.js"
script_path.write_text(script, encoding="utf-8")

result = subprocess.run(
    ["node", str(script_path), str(CONTRACT_FILE)],
    cwd=str(LEGACY_DIR),
    capture_output=True,
    text=True,
    timeout=120
)

print("========== STDOUT ==========")
print(result.stdout[:8000])

print("========== STDERR ==========")
print(result.stderr[:3000])

print("Exit code:", result.returncode)

========== STDOUT ==========
COMPILER: 0.4.8+commit.60cc1668.Emscripten.clang
CONTRACTS_FOUND: 0

========== STDERR ==========
(node:53636) V8: soljson-v0.4.8+commit.60cc1668.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)

Exit code: 2


In [40]:
LEGACY_DIR = Path("/content/solc_legacy")
CONTRACT_FILE = Path("/content/solidity_data/C002244.sol")

if not CONTRACT_FILE.exists():
    matches = list(
        Path("/content/solidity_data").rglob("C002244.sol")
    )
    if not matches:
        raise FileNotFoundError("Không tìm thấy C002244.sol")
    CONTRACT_FILE = matches[0]

script = r"""
const solc = require('solc');
const fs = require('fs');

const source = fs.readFileSync(process.argv[2], 'utf8');

solc.loadRemoteVersion(
    'v0.4.8+commit.60cc1668',
    (error, compiler) => {

        if (error) {
            console.error('LOAD_ERROR:', error.message);
            process.exitCode = 1;
            return;
        }

        console.log('COMPILER:', compiler.version());
        console.log('SOURCE_LENGTH:', source.length);

        try {
            const output = compiler.compile(source, 0);

            console.log('OUTPUT_TYPE:', typeof output);

            if (typeof output === 'string') {
                console.log('OUTPUT_PREVIEW:', output.slice(0, 4000));

                try {
                    const parsed = JSON.parse(output);
                    console.log('PARSED_KEYS:', Object.keys(parsed));
                    console.log(
                        'PARSED_ERRORS:',
                        JSON.stringify(parsed.errors || []).slice(0, 4000)
                    );
                } catch (e) {
                    console.log('JSON_PARSE_ERROR:', e.message);
                }

            } else if (output && typeof output === 'object') {
                console.log('OUTPUT_KEYS:', Object.keys(output));

                console.log(
                    'ERRORS:',
                    JSON.stringify(output.errors || []).slice(0, 4000)
                );

                console.log(
                    'CONTRACT_KEYS:',
                    Object.keys(output.contracts || {})
                );

            } else {
                console.log('UNEXPECTED_OUTPUT:', String(output));
            }

        } catch (e) {
            console.error('COMPILE_EXCEPTION:', e.stack);
            process.exitCode = 1;
        }
    }
);
"""

script_path = LEGACY_DIR / "diagnose_legacy.js"
script_path.write_text(script, encoding="utf-8")

result = subprocess.run(
    ["node", str(script_path), str(CONTRACT_FILE)],
    cwd=str(LEGACY_DIR),
    capture_output=True,
    text=True,
    timeout=120
)

print("========== STDOUT ==========")
print(result.stdout[:10000])

print("========== STDERR ==========")
print(result.stderr[:3000])

print("Exit code:", result.returncode)

========== STDOUT ==========
COMPILER: 0.4.8+commit.60cc1668.Emscripten.clang
SOURCE_LENGTH: 3379
OUTPUT_TYPE: string
OUTPUT_PREVIEW: {"errors":[{"type":"JSONError","component":"solcjs","severity":"error","message":"Invalid JSON supplied: Unexpected token 'p', \"pragma sol\"... is not valid JSON","formattedMessage":"Error: Invalid JSON supplied: Unexpected token 'p', \"pragma sol\"... is not valid JSON"}]}
PARSED_KEYS: [ 'errors' ]
PARSED_ERRORS: [{"type":"JSONError","component":"solcjs","severity":"error","message":"Invalid JSON supplied: Unexpected token 'p', \"pragma sol\"... is not valid JSON","formattedMessage":"Error: Invalid JSON supplied: Unexpected token 'p', \"pragma sol\"... is not valid JSON"}]

========== STDERR ==========
(node:53684) V8: soljson-v0.4.8+commit.60cc1668.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)

Exit code: 0


In [41]:
LEGACY_DIR = Path("/content/solc_legacy")

CONTRACT_FILE = Path("/content/solidity_data/C002244.sol")

if not CONTRACT_FILE.exists():
    matches = list(Path("/content/solidity_data").rglob("C002244.sol"))
    if not matches:
        raise FileNotFoundError("Không tìm thấy C002244.sol")
    CONTRACT_FILE = matches[0]

script = r"""
const solc = require('solc');
const fs = require('fs');

const source = fs.readFileSync(process.argv[2], 'utf8');

solc.loadRemoteVersion(
    'v0.4.8+commit.60cc1668',
    (error, compiler) => {

        if (error) {
            console.error('LOAD_ERROR:', error.message);
            process.exitCode = 1;
            return;
        }

        console.log('COMPILER:', compiler.version());

        const input = {
            language: 'Solidity',
            sources: {
                'C002244.sol': {
                    content: source
                }
            },
            settings: {
                optimizer: {
                    enabled: false
                },
                outputSelection: {
                    '*': {
                        '*': [
                            'evm.bytecode.object',
                            'evm.deployedBytecode.object'
                        ]
                    }
                }
            }
        };

        try {
            const output = JSON.parse(
                compiler.compile(JSON.stringify(input))
            );

            if (output.errors) {
                console.log('========== COMPILER MESSAGES ==========');

                for (const err of output.errors) {
                    console.log(
                        err.severity + ': ' +
                        (err.formattedMessage || err.message)
                    );
                }
            }

            const contracts = output.contracts || {};

            let count = 0;

            for (const fileName of Object.keys(contracts)) {
                for (const contractName of Object.keys(contracts[fileName])) {

                    const artifact = contracts[fileName][contractName];

                    const creation =
                        artifact.evm?.bytecode?.object || '';

                    const runtime =
                        artifact.evm?.deployedBytecode?.object || '';

                    console.log('CONTRACT:', contractName);
                    console.log(
                        'CREATION_BYTECODE_LENGTH:',
                        creation.length / 2
                    );
                    console.log(
                        'RUNTIME_BYTECODE_LENGTH:',
                        runtime.length / 2
                    );

                    count++;
                }
            }

            console.log('CONTRACTS_FOUND:', count);

            if (count === 0) {
                process.exitCode = 2;
            }

        } catch (e) {
            console.error('COMPILE_EXCEPTION:', e.stack);
            process.exitCode = 1;
        }
    }
);
"""

script_path = LEGACY_DIR / "compile_legacy_fixed.js"
script_path.write_text(script, encoding="utf-8")

result = subprocess.run(
    ["node", str(script_path), str(CONTRACT_FILE)],
    cwd=str(LEGACY_DIR),
    capture_output=True,
    text=True,
    timeout=120
)

print("========== STDOUT ==========")
print(result.stdout[:10000])

print("========== STDERR ==========")
print(result.stderr[:3000])

print("Exit code:", result.returncode)

========== STDOUT ==========
COMPILER: 0.4.8+commit.60cc1668.Emscripten.clang
========== COMPILER MESSAGES ==========
CONTRACT: ERC20TokenInterface
CREATION_BYTECODE_LENGTH: 0
RUNTIME_BYTECODE_LENGTH: 0
CONTRACT: Gidnist
CREATION_BYTECODE_LENGTH: 4942
RUNTIME_BYTECODE_LENGTH: 4531
CONTRACTS_FOUND: 2

========== STDERR ==========
(node:53804) V8: soljson-v0.4.8+commit.60cc1668.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)

Exit code: 0


In [42]:
LEGACY_DIR = Path("/content/solc_legacy")

script = r"""
const solc = require('solc');

const versions = [
    'v0.4.4+commit.4633f3de',
    'v0.4.6+commit.2dabbdf0',
    'v0.4.9+commit.364da425',
    'v0.4.10+commit.f0d539ae'
];

async function main() {
    for (const version of versions) {
        console.log('\nTESTING:', version);

        await new Promise(resolve => {
            solc.loadRemoteVersion(version, (error, compiler) => {
                if (error) {
                    console.log('FAILED:', error.message);
                } else {
                    console.log('SUCCESS:', compiler.version());
                }
                resolve();
            });
        });
    }
}

main().catch(error => {
    console.error(error);
    process.exitCode = 1;
});
"""

script_path = LEGACY_DIR / "test_remaining_versions.js"
script_path.write_text(script, encoding="utf-8")

result = subprocess.run(
    ["node", str(script_path)],
    cwd=str(LEGACY_DIR),
    capture_output=True,
    text=True,
    timeout=240
)

print("========== STDOUT ==========")
print(result.stdout)

print("========== STDERR ==========")
print(result.stderr[:3000])

print("Exit code:", result.returncode)

========== STDOUT ==========

TESTING: v0.4.4+commit.4633f3de
SUCCESS: 0.4.4+commit.4633f3de.Emscripten.clang

TESTING: v0.4.6+commit.2dabbdf0
SUCCESS: 0.4.6+commit.2dabbdf0.Emscripten.clang

TESTING: v0.4.9+commit.364da425
SUCCESS: 0.4.9+commit.364da425.Emscripten.clang

TESTING: v0.4.10+commit.f0d539ae
SUCCESS: 0.4.10+commit.f0d539ae.Emscripten.clang

========== STDERR ==========
(node:53938) V8: soljson-v0.4.4+commit.4633f3de.js:3 Invalid asm.js: Invalid member of stdlib
(Use `node --trace-warnings ...` to show where the warning was created)
(node:53938) V8: soljson-v0.4.6+commit.2dabbdf0.js:3 Invalid asm.js: Invalid member of stdlib
(node:53938) V8: soljson-v0.4.9+commit.364da425.js:3 Invalid asm.js: Invalid member of stdlib
(node:53938) V8: soljson-v0.4.10+commit.f0d539ae.js:3 Invalid asm.js: Invalid member of stdlib

Exit code: 0


In [43]:
# Lấy 20 file đầu tiên không có pragma
no_pragma_ids = df_compiler_mapping.loc[
    df_compiler_mapping["status"] == "NO_PRAGMA",
    "contract_id"
].head(20).tolist()

# Những compiler sẽ thử
test_versions = [
    "0.4.26",
    "0.4.24",
    "0.5.17",
    "0.6.12",
    "0.7.6"
]

# Tạo ánh xạ contract_id -> đường dẫn file
file_lookup = {f.stem: f for f in sol_files}

results = []
start_time = time.time()

for contract_id in no_pragma_ids:
    file = file_lookup[contract_id]
    source = file.read_text(encoding="utf-8", errors="replace")

    print(f"Đang kiểm tra {contract_id}...", flush=True)

    success_version = None
    errors = []

    for version in test_versions:
        try:
            compiled = solcx.compile_source(
                source,
                output_values=["bin", "bin-runtime"],
                solc_version=version
            )

            # Ưu tiên file có runtime bytecode
            has_runtime = any(
                artifact.get("bin-runtime")
                for artifact in compiled.values()
            )

            if has_runtime:
                success_version = version
                break

            errors.append(f"{version}: NO_RUNTIME_BYTECODE")

        except Exception as e:
            errors.append(f"{version}: {str(e)[:150]}")

    results.append({
        "contract_id": contract_id,
        "selected_compiler": success_version,
        "status": "SUCCESS" if success_version else "FAILED",
        "errors": " | ".join(errors)
    })

df_no_pragma_test = pd.DataFrame(results)

print("\n========== KẾT QUẢ ==========")
print(df_no_pragma_test["status"].value_counts())

print("\n========== COMPILER THÀNH CÔNG ==========")
print(df_no_pragma_test["selected_compiler"].value_counts())

print("\n========== FILE CHƯA THÀNH CÔNG ==========")
display(
    df_no_pragma_test[
        df_no_pragma_test["status"] == "FAILED"
    ][["contract_id", "errors"]]
)

print(f"\nThời gian: {time.time() - start_time:.2f} giây")

Đang kiểm tra C000564...
Đang kiểm tra C000616...
Đang kiểm tra C000632...
Đang kiểm tra C000709...
Đang kiểm tra C000712...
Đang kiểm tra C000786...
Đang kiểm tra C000794...
Đang kiểm tra C000795...
Đang kiểm tra C000826...
Đang kiểm tra C000832...
Đang kiểm tra C000917...
Đang kiểm tra C000918...
Đang kiểm tra C000921...
Đang kiểm tra C001132...
Đang kiểm tra C001165...
Đang kiểm tra C001172...
Đang kiểm tra C001179...
Đang kiểm tra C001201...
Đang kiểm tra C001322...
Đang kiểm tra C001346...

========== KẾT QUẢ ==========
status
SUCCESS    20
Name: count, dtype: int64

========== COMPILER THÀNH CÔNG ==========
selected_compiler
0.4.26    20
Name: count, dtype: int64

========== FILE CHƯA THÀNH CÔNG ==========


,contract_id,errors



Thời gian: 0.91 giây


In [ ]:
import solcx
import pandas as pd
import time
from pathlib import Path

# Thư mục lưu kết quả lâu dài
OUTPUT_DIR = Path("/content/drive/MyDrive/solidity_experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "no_pragma_compilation.csv"

# Danh sách file không có pragma
no_pragma_ids = df_compiler_mapping.loc[
    df_compiler_mapping["status"] == "NO_PRAGMA",
    "contract_id"
].tolist()

file_lookup = {f.stem: f for f in sol_files}

# Đọc checkpoint nếu đã chạy trước đó
if CHECKPOINT_PATH.exists():
    previous = pd.read_csv(
        CHECKPOINT_PATH,
        keep_default_na=False
    )
    results = previous.to_dict("records")
    completed = set(previous["contract_id"])
else:
    results = []
    completed = set()

print("Tổng file cần xử lý:", len(no_pragma_ids))
print("Đã xử lý trước đó:", len(completed))

start_time = time.time()

for index, contract_id in enumerate(no_pragma_ids, start=1):

    if contract_id in completed:
        continue

    file = file_lookup[contract_id]

    try:
        source = file.read_text(
            encoding="utf-8",
            errors="replace"
        )

        compiled = solcx.compile_source(
            source,
            output_values=["bin", "bin-runtime"],
            solc_version="0.4.26"
        )

        contract_count = len(compiled)

        runtime_count = sum(
            bool(artifact.get("bin-runtime"))
            for artifact in compiled.values()
        )

        runtime_bytes = sum(
            len(artifact.get("bin-runtime", "")) // 2
            for artifact in compiled.values()
        )

        status = (
            "SUCCESS"
            if runtime_count > 0
            else "NO_RUNTIME_BYTECODE"
        )

        error = ""

    except Exception as e:
        contract_count = 0
        runtime_count = 0
        runtime_bytes = 0
        status = "FAILED"
        error = str(e)[:1000]

    results.append({
        "contract_id": contract_id,
        "compiler": "0.4.26",
        "status": status,
        "contract_count": contract_count,
        "runtime_contract_count": runtime_count,
        "runtime_bytes": runtime_bytes,
        "error": error
    })

    completed.add(contract_id)

    if index % 100 == 0:
        pd.DataFrame(results).to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        print(
            f"Đã xử lý {index}/{len(no_pragma_ids)} file",
            flush=True
        )

# Lưu kết quả cuối cùng
df_no_pragma_compilation = pd.DataFrame(results)

df_no_pragma_compilation.to_csv(
    CHECKPOINT_PATH,
    index=False
)

print("\n========== KẾT QUẢ ==========")
print(df_no_pragma_compilation["status"].value_counts())

print("\n========== THỐNG KÊ ==========")
print("Tổng file:", len(df_no_pragma_compilation))
print(
    "Có Runtime Bytecode:",
    (df_no_pragma_compilation["status"] == "SUCCESS").sum()
)

print(
    "Không có Runtime Bytecode:",
    (df_no_pragma_compilation["status"] == "NO_RUNTIME_BYTECODE").sum()
)

print(
    "Biên dịch thất bại:",
    (df_no_pragma_compilation["status"] == "FAILED").sum()
)

print(
    f"\nThời gian: {(time.time() - start_time):.2f} giây"
)

print("Checkpoint:", CHECKPOINT_PATH)

# **COMPILE CONTRACTS**

In [44]:
EXTRACT_DIR = Path("/content/solidity_data")
CSV_PATH = Path("/content/drive/MyDrive/IS6002/EXPERIMENT/implementation/data/contracts_and_label.csv")
LEGACY_DIR = Path("/content/solc_legacy")

sol_files = sorted(EXTRACT_DIR.rglob("*.sol"))

print("========== KIỂM TRA DỮ LIỆU ==========")
print("Số file Solidity:", len(sol_files))
print("File CSV tồn tại:", CSV_PATH.exists())
print("Thư mục solc-js tồn tại:", LEGACY_DIR.exists())
print(
    "solc-js đã cài:",
    (LEGACY_DIR / "node_modules" / "solc").exists()
)

print("\n========== KIỂM TRA COMPILER ==========")
print("Số compiler py-solc-x:", len(solcx.get_installed_solc_versions()))

print("\n========== KIỂM TRA MAPPING ==========")
print("Số contract trong mapping:", len(df_compiler_mapping))
print(df_compiler_mapping["status"].value_counts())

assert len(sol_files) == 39905, "Thiếu file Solidity!"
assert CSV_PATH.exists(), "Không tìm thấy file nhãn!"
assert len(df_compiler_mapping) == 39905, "Mapping chưa đầy đủ!"

print("\nKIỂM TRA HOÀN TẤT")

========== KIỂM TRA DỮ LIỆU ==========
Số file Solidity: 39905
File CSV tồn tại: True
Thư mục solc-js tồn tại: True
solc-js đã cài: True

========== KIỂM TRA COMPILER ==========
Số compiler py-solc-x: 17

========== KIỂM TRA MAPPING ==========
Số contract trong mapping: 39905
status
COMPILER_AVAILABLE        38793
NO_PRAGMA                  1059
NO_COMPATIBLE_COMPILER       53
Name: count, dtype: int64

KIỂM TRA HOÀN TẤT


In [45]:
def extract_opcode_counts(bytecode_hex):
    """
    Đếm Opcode từ EVM Bytecode.

    - Bỏ qua dữ liệu theo sau PUSH1..PUSH32
    - Giữ nguyên các Opcode chưa biết dưới dạng OP_XX
    - Không tự động loại bỏ metadata
    """

    bytecode_hex = bytecode_hex.removeprefix("0x")
    bytecode = bytes.fromhex(bytecode_hex)

    counts = Counter()
    i = 0

    while i < len(bytecode):
        opcode = bytecode[i]

        if 0x60 <= opcode <= 0x7F:
            push_size = opcode - 0x5F
            name = f"PUSH{push_size}"
            i += 1 + push_size

        else:
            name = f"OP_{opcode:02X}"
            i += 1

        counts[name] += 1

    return counts


test_bytecode = "6080604052"

result = extract_opcode_counts(test_bytecode)

print("========== KẾT QUẢ KIỂM THỬ ==========")
print(result)
print("Tổng số Opcode:", sum(result.values()))

assert result["PUSH1"] == 2
assert result["OP_52"] == 1
assert sum(result.values()) == 3

print("KIỂM THỬ THÀNH CÔNG")

========== KẾT QUẢ KIỂM THỬ ==========
Counter({'PUSH1': 2, 'OP_52': 1})
Tổng số Opcode: 3
KIỂM THỬ THÀNH CÔNG


In [48]:
import json
import subprocess
import solcx
from pathlib import Path

LEGACY_DIR = Path("/content/solc_legacy")
file_lookup = {f.stem: f for f in sol_files}

legacy_versions = {
    "0.4.4": "v0.4.4+commit.4633f3de",
    "0.4.6": "v0.4.6+commit.2dabbdf0",
    "0.4.8": "v0.4.8+commit.60cc1668",
    "0.4.9": "v0.4.9+commit.364da425",
    "0.4.10": "v0.4.10+commit.f0d539ae"
}

# Script solc-js: biên dịch bằng Standard JSON
legacy_script = r"""
const solc = require('solc');
const fs = require('fs');

const filePath = process.argv[2];
const version = process.argv[3];
const source = fs.readFileSync(filePath, 'utf8');

solc.loadRemoteVersion(version, (error, compiler) => {
    if (error) {
        console.error('LOAD_ERROR:', error.message);
        process.exitCode = 1;
        return;
    }

    const input = {
        language: 'Solidity',
        sources: {
            [require('path').basename(filePath)]: {
                content: source
            }
        },
        settings: {
            optimizer: { enabled: false },
            outputSelection: {
                '*': {
                    '*': [
                        'evm.bytecode.object',
                        'evm.deployedBytecode.object'
                    ]
                }
            }
        }
    };

    try {
        const output = JSON.parse(
            compiler.compile(JSON.stringify(input))
        );

        const errors = (output.errors || [])
            .filter(e => e.severity === 'error')
            .map(e => e.formattedMessage || e.message);

        const runtimes = {};

        for (const [file, contracts] of Object.entries(output.contracts || {})) {
            for (const [name, artifact] of Object.entries(contracts)) {
                const runtime =
                    artifact.evm?.deployedBytecode?.object || '';

                runtimes[name] = runtime;
            }
        }

        console.log(JSON.stringify({
            compiler: compiler.version(),
            errors,
            runtimes
        }));

    } catch (e) {
        console.error('COMPILE_ERROR:', e.message);
        process.exitCode = 1;
    }
});
"""

script_path = LEGACY_DIR / "compile_for_opcode.js"
script_path.write_text(legacy_script, encoding="utf-8")


def compile_and_extract(contract_id, compiler_version):
    file = file_lookup[contract_id]

    # 1. Biên dịch
    if compiler_version in legacy_versions:
        result = subprocess.run(
            [
                "node",
                str(script_path),
                str(file),
                legacy_versions[compiler_version]
            ],
            cwd=str(LEGACY_DIR),
            capture_output=True,
            text=True,
            timeout=120
        )

        if result.returncode != 0:
            raise RuntimeError(result.stderr[:1000])

        output = json.loads(result.stdout)
        if output["errors"]:
            raise RuntimeError("; ".join(output["errors"]))

        runtimes = output["runtimes"]

    else:
        source = file.read_text(
            encoding="utf-8",
            errors="replace"
        )

        compiled = solcx.compile_source(
            source,
            output_values=["bin-runtime"],
            solc_version=compiler_version
        )

        runtimes = {
            name: artifact.get("bin-runtime", "")
            for name, artifact in compiled.items()
        }

    # 2. Trích xuất Opcode
    from collections import Counter
    total_counts = Counter()
    runtime_contracts = 0

    for name, runtime in runtimes.items():
        if not runtime:
            continue

        counts = extract_opcode_counts(runtime)
        total_counts.update(counts)
        runtime_contracts += 1

    return {
        "contract_id": contract_id,
        "compiler": compiler_version,
        "runtime_contracts": runtime_contracts,
        "total_opcodes": sum(total_counts.values()),
        "opcode_counts": dict(total_counts)
    }


# Chọn 3 file thử nghiệm
sample_05 = df_compiler_mapping.loc[
    df_compiler_mapping["selected_compiler"] == "0.5.17",
    "contract_id"
].iloc[0]

test_cases = [
    ("C002244", "0.4.8"),
    ("C000564", "0.4.26"),
    (sample_05, "0.5.17")
]

for contract_id, compiler in test_cases:
    print(f"\nĐang xử lý {contract_id} bằng solc {compiler}")

    try:
        result = compile_and_extract(contract_id, compiler)

        print("Runtime contracts:", result["runtime_contracts"])
        print("Tổng Opcode:", result["total_opcodes"])
        print(
            "10 Opcode phổ biến:",
            sorted(
                result["opcode_counts"].items(),
                key=lambda x: x[1],
                reverse=True
            )[:10]
        )

    except Exception as e:
        print("FAILED:", str(e)[:1000])


Đang xử lý C002244 bằng solc 0.4.8
Runtime contracts: 1
Tổng Opcode: 2328
10 Opcode phổ biến: [('PUSH1', 329), ('OP_50', 181), ('OP_90', 180), ('OP_80', 148), ('OP_01', 135), ('OP_81', 133), ('PUSH2', 128), ('OP_5B', 95), ('OP_52', 92), ('OP_91', 86)]

Đang xử lý C000564 bằng solc 0.4.26
Runtime contracts: 1
Tổng Opcode: 1291
10 Opcode phổ biến: [('PUSH1', 186), ('OP_80', 112), ('OP_50', 93), ('OP_90', 83), ('OP_81', 78), ('OP_01', 76), ('PUSH2', 74), ('OP_5B', 58), ('OP_52', 49), ('OP_91', 45)]

Đang xử lý C000001 bằng solc 0.5.17
Runtime contracts: 1
Tổng Opcode: 3359
10 Opcode phổ biến: [('PUSH1', 481), ('OP_50', 277), ('OP_90', 257), ('OP_80', 248), ('OP_81', 215), ('OP_01', 180), ('PUSH2', 174), ('OP_5B', 153), ('OP_52', 109), ('OP_16', 109)]


In [49]:
test_cases = [
    ("C002244", "0.4.8"),
    ("C000564", "0.4.26"),
    ("C000001", "0.5.17")
]

def inspect_runtime(contract_id, compiler_version):
    file = file_lookup[contract_id]

    if compiler_version in legacy_versions:
        import subprocess
        import json

        result = subprocess.run(
            [
                "node",
                str(script_path),
                str(file),
                legacy_versions[compiler_version]
            ],
            cwd=str(LEGACY_DIR),
            capture_output=True,
            text=True,
            timeout=120,
            check=True
        )

        output = json.loads(result.stdout)

        if output.get("errors"):
            raise RuntimeError("; ".join(output["errors"]))

        runtimes = output["runtimes"]

    else:
        compiled = solcx.compile_source(
            file.read_text(encoding="utf-8", errors="replace"),
            output_values=["bin-runtime"],
            solc_version=compiler_version
        )

        runtimes = {
            name: artifact.get("bin-runtime", "")
            for name, artifact in compiled.items()
        }

    for name, runtime in runtimes.items():
        if not runtime:
            continue

        is_hex = bool(re.fullmatch(r"[0-9a-fA-F]+", runtime))
        is_even = len(runtime) % 2 == 0

        print(f"\nContract ID: {contract_id}")
        print(f"Contract name: {name}")
        print(f"Compiler: {compiler_version}")
        print(f"Runtime length: {len(runtime) // 2} bytes")
        print(f"Valid hex: {is_hex and is_even}")
        print(f"Last 100 hex characters: {runtime[-100:]}")

        if not is_hex or not is_even:
            print("WARNING: Bytecode không phải chuỗi hex hợp lệ")

for contract_id, compiler in test_cases:
    inspect_runtime(contract_id, compiler)


Contract ID: C002244
Contract name: Gidnist
Compiler: 0.4.8
Runtime length: 4531 bytes
Valid hex: True
Last 100 hex characters: 5b929150505600a165627a7a723058206ad871fdb815b9051f4537a4d6458007692db414f56fb1adad239994b9fd71290029

Contract ID: C000564
Contract name: <stdin>:InfraCoin
Compiler: 0.4.26
Runtime length: 2006 bytes
Valid hex: True
Last 100 hex characters: 91505054815600a165627a7a72305820b3ca0a4caecda97ef52c636137234196585dbb2b571ef2e031792b558da82e5b0029

Contract ID: C000001
Contract name: <stdin>:HotDollarsToken
Compiler: 0.5.17
Runtime length: 6218 bytes
Valid hex: True
Last 100 hex characters: 627a7a72315820581ba54df3c582a4cd86972140d49b4a8043fbdfeb6104022c1a789cbc1eba1164736f6c63430005110032


In [50]:
import io
import sys

try:
    import cbor2
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "cbor2"
    ])
    import cbor2


# 2. Hàm loại bỏ Solidity metadata
def strip_solidity_metadata(bytecode_hex):

    bytecode_hex = bytecode_hex.removeprefix("0x")

    if not re.fullmatch(r"[0-9a-fA-F]*", bytecode_hex):
        raise ValueError("Bytecode không phải hex hợp lệ")

    if len(bytecode_hex) % 2 != 0:
        raise ValueError("Độ dài Bytecode không hợp lệ")

    bytecode = bytes.fromhex(bytecode_hex)

    if len(bytecode) < 4:
        return bytecode_hex, False, 0

    # Hai byte cuối chứa độ dài metadata
    metadata_length = int.from_bytes(
        bytecode[-2:], byteorder="big"
    )

    metadata_start = len(bytecode) - 2 - metadata_length

    if metadata_length <= 0 or metadata_start < 0:
        return bytecode_hex, False, 0

    metadata = bytecode[metadata_start:-2]

    try:
        stream = io.BytesIO(metadata)
        decoded = cbor2.load(stream)

        # Kiểm tra metadata hợp lệ
        if stream.tell() != len(metadata):
            return bytecode_hex, False, 0

        known_keys = {
            "bzzr0", "bzzr1", "bzzr",
            "ipfs", "solc", "experimental"
        }

        if not isinstance(decoded, dict):
            return bytecode_hex, False, 0

        if not any(key in decoded for key in known_keys):
            return bytecode_hex, False, 0

        clean_bytecode = bytecode[:metadata_start]

        return clean_bytecode.hex(), True, metadata_length + 2

    except Exception:
        return bytecode_hex, False, 0


# 3. Hàm lấy Runtime Bytecode
def get_runtime_bytecodes(contract_id, compiler_version):

    file = file_lookup[contract_id]

    if compiler_version in legacy_versions:

        result = subprocess.run(
            [
                "node",
                str(script_path),
                str(file),
                legacy_versions[compiler_version]
            ],
            cwd=str(LEGACY_DIR),
            capture_output=True,
            text=True,
            timeout=120,
            check=True
        )

        output = json.loads(result.stdout)

        if output.get("errors"):
            raise RuntimeError("; ".join(output["errors"]))

        return output["runtimes"]

    compiled = solcx.compile_source(
        file.read_text(encoding="utf-8", errors="replace"),
        output_values=["bin-runtime"],
        solc_version=compiler_version
    )

    return {
        name: artifact.get("bin-runtime", "")
        for name, artifact in compiled.items()
    }


# 4. Kiểm tra trên 3 file đã biên dịch thành công
test_cases = [
    ("C002244", "0.4.8"),
    ("C000564", "0.4.26"),
    ("C000001", "0.5.17")
]

print("========== KIỂM TRA METADATA ==========")

for contract_id, compiler in test_cases:

    runtimes = get_runtime_bytecodes(
        contract_id, compiler
    )

    for name, runtime in runtimes.items():

        if not runtime:
            continue

        clean, removed, metadata_bytes = (
            strip_solidity_metadata(runtime)
        )

        original_counts = extract_opcode_counts(runtime)
        clean_counts = extract_opcode_counts(clean)

        print(f"\nContract ID: {contract_id}")
        print(f"Compiler: {compiler}")
        print(f"Metadata detected: {removed}")
        print(f"Metadata removed: {metadata_bytes} bytes")
        print(
            "Original Opcode count:",
            sum(original_counts.values())
        )
        print(
            "Clean Opcode count:",
            sum(clean_counts.values())
        )
        print(
            "Opcode difference:",
            sum(original_counts.values()) -
            sum(clean_counts.values())
        )

print("\n========== HOÀN TẤT KIỂM TRA ==========")

========== KIỂM TRA METADATA ==========

Contract ID: C002244
Compiler: 0.4.8
Metadata detected: True
Metadata removed: 43 bytes
Original Opcode count: 2328
Clean Opcode count: 2315
Opcode difference: 13

Contract ID: C000564
Compiler: 0.4.26
Metadata detected: True
Metadata removed: 43 bytes
Original Opcode count: 1291
Clean Opcode count: 1280
Opcode difference: 11

Contract ID: C000001
Compiler: 0.5.17
Metadata detected: True
Metadata removed: 52 bytes
Original Opcode count: 3359
Clean Opcode count: 3331
Opcode difference: 28

========== HOÀN TẤT KIỂM TRA ==========


In [51]:
sample_rows = df_compiler_mapping.head(100)
results = []

start = time.time()

for _, row in sample_rows.iterrows():
    contract_id = row["contract_id"]
    compiler = row["selected_compiler"]

    try:
        result = compile_and_extract(contract_id, compiler)

        results.append({
            "contract_id": contract_id,
            "compiler": compiler,
            "status": "SUCCESS" if result["total_opcodes"] > 0 else "NO_RUNTIME",
            "runtime_contracts": result["runtime_contracts"],
            "total_opcodes": result["total_opcodes"]
        })

    except Exception as e:
        results.append({
            "contract_id": contract_id,
            "compiler": compiler,
            "status": "FAILED",
            "runtime_contracts": 0,
            "total_opcodes": 0,
            "error": str(e)[:200]
        })

df_test_100 = pd.DataFrame(results)

print("========== KẾT QUẢ 100 FILE ==========")
print(df_test_100["status"].value_counts())

print("\nCác file thất bại:")
print(
    df_test_100[df_test_100["status"] == "FAILED"]
    .to_string(index=False)
)

print(f"\nThời gian: {time.time() - start:.2f} giây")

========== KẾT QUẢ 100 FILE ==========
status
SUCCESS    84
FAILED     16
Name: count, dtype: int64

Các file thất bại:
contract_id compiler status  runtime_contracts  total_opcodes                                                                                                                                                                                                          error
    C000010    0.7.6 FAILED                  0              0 An error occurred during execution\n> command: `/root/.solcx/solc-v0.7.6 --combined-json bin-runtime -`\n> return code: `1`\n> stdout:\n\n> stderr:\nExpected a state variable declaration. If you intended th
    C000015    0.7.6 FAILED                  0              0 An error occurred during execution\n> command: `/root/.solcx/solc-v0.7.6 --combined-json bin-runtime -`\n> return code: `1`\n> stdout:\n\n> stderr:\nExpected a state variable declaration. If you intended th
    C000022    0.7.6 FAILED                  0              0 An error oc

In [52]:
failed_ids = (
    df_test_100.loc[
        df_test_100["status"] == "FAILED",
        "contract_id"
    ].head(5).tolist()
)

for contract_id in failed_ids:

    file = file_lookup[contract_id]
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    mapping = df_compiler_mapping.loc[
        df_compiler_mapping["contract_id"] == contract_id
    ].iloc[0]

    compiler = mapping["selected_compiler"]

    pragmas = re.findall(
        r"pragma\s+solidity\s+([^;]+);",
        source
    )

    print("\n" + "=" * 65)
    print("CONTRACT ID:", contract_id)
    print("PRAGMA:", pragmas)
    print("COMPILER ĐÃ CHỌN:", compiler)

    try:
        solcx.compile_source(
            source,
            output_values=["bin-runtime"],
            solc_version=compiler
        )

        print("STATUS: SUCCESS")

    except Exception as e:
        error = str(e)

        # Lấy phần thông báo lỗi thực tế
        if "> stderr:" in error:
            error = error.split("> stderr:", 1)[1]

        print("STATUS: FAILED")
        print("CHI TIẾT LỖI:")
        print(error[:3000])

print("\n========== HOÀN TẤT KIỂM TRA ==========")


CONTRACT ID: C000010
PRAGMA: ['>=0.5.9']
COMPILER ĐÃ CHỌN: 0.7.6
STATUS: FAILED
CHI TIẾT LỖI:

Expected a state variable declaration. If you intended this as a fallback function or a function to handle plain ether transactions, use the "fallback" keyword or the "receive" keyword instead.
   --> <stdin>:650:34:
    |
650 |     function () external payable {
    |                                  ^

CONTRACT ID: C000015
PRAGMA: ['>=0.5.1']
COMPILER ĐÃ CHỌN: 0.7.6
STATUS: FAILED
CHI TIẾT LỖI:

Expected a state variable declaration. If you intended this as a fallback function or a function to handle plain ether transactions, use the "fallback" keyword or the "receive" keyword instead.
  --> <stdin>:70:34:
   |
70 |     function () payable external {
   |                                  ^

CONTRACT ID: C000022
PRAGMA: ['>=0.5.11']
COMPILER ĐÃ CHỌN: 0.7.6
STATUS: FAILED
CHI TIẾT LỖI:

--> <stdin>

   --> <stdin>:325:13:
    |
325 |             uint256 id = listingIds[i];
    |             

In [53]:
failed_ids = (
    df_test_100.loc[
        df_test_100["status"] == "FAILED",
        "contract_id"
    ].head(5).tolist()
)

for contract_id in failed_ids:

    file = file_lookup[contract_id]
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    mapping = df_compiler_mapping.loc[
        df_compiler_mapping["contract_id"] == contract_id
    ].iloc[0]

    compiler = mapping["selected_compiler"]

    pragmas = re.findall(
        r"pragma\s+solidity\s+([^;]+);",
        source
    )

    print("\n" + "=" * 65)
    print("CONTRACT ID:", contract_id)
    print("PRAGMA:", pragmas)
    print("COMPILER ĐÃ CHỌN:", compiler)

    try:
        solcx.compile_source(
            source,
            output_values=["bin-runtime"],
            solc_version=compiler
        )

        print("STATUS: SUCCESS")

    except Exception as e:
        error = str(e)

        # Lấy phần thông báo lỗi thực tế
        if "> stderr:" in error:
            error = error.split("> stderr:", 1)[1]

        print("STATUS: FAILED")
        print("CHI TIẾT LỖI:")
        print(error[:3000])

print("\n========== HOÀN TẤT KIỂM TRA ==========")


CONTRACT ID: C000010
PRAGMA: ['>=0.5.9']
COMPILER ĐÃ CHỌN: 0.7.6
STATUS: FAILED
CHI TIẾT LỖI:

Expected a state variable declaration. If you intended this as a fallback function or a function to handle plain ether transactions, use the "fallback" keyword or the "receive" keyword instead.
   --> <stdin>:650:34:
    |
650 |     function () external payable {
    |                                  ^

CONTRACT ID: C000015
PRAGMA: ['>=0.5.1']
COMPILER ĐÃ CHỌN: 0.7.6
STATUS: FAILED
CHI TIẾT LỖI:

Expected a state variable declaration. If you intended this as a fallback function or a function to handle plain ether transactions, use the "fallback" keyword or the "receive" keyword instead.
  --> <stdin>:70:34:
   |
70 |     function () payable external {
   |                                  ^

CONTRACT ID: C000022
PRAGMA: ['>=0.5.11']
COMPILER ĐÃ CHỌN: 0.7.6
STATUS: FAILED
CHI TIẾT LỖI:

--> <stdin>

   --> <stdin>:325:13:
    |
325 |             uint256 id = listingIds[i];
    |             

In [54]:
import semantic_version

# Danh sách compiler đã cài
installed_versions = sorted(
    [str(v) for v in solcx.get_installed_solc_versions()],
    key=semantic_version.Version
)

def compiler_matches_pragma(source, version):
    pragmas = re.findall(
        r"pragma\s+solidity\s+([^;]+);",
        source
    )

    if not pragmas:
        return True

    try:
        compiler_ver = semantic_version.Version(version)

        return all(
            semantic_version.NpmSpec(p.strip()).match(compiler_ver)
            for p in pragmas
        )
    except Exception:
        return False


failed_ids = df_test_100.loc[
    df_test_100["status"] == "FAILED",
    "contract_id"
].tolist()

retry_results = []

for contract_id in failed_ids:

    file = file_lookup[contract_id]
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    compatible = [
        v for v in installed_versions
        if compiler_matches_pragma(source, v)
    ]

    selected = None
    runtime_count = 0
    last_error = ""

    for version in compatible:
        try:
            compiled = solcx.compile_source(
                source,
                output_values=["bin-runtime"],
                solc_version=version
            )

            count = sum(
                bool(item.get("bin-runtime"))
                for item in compiled.values()
            )

            if count > 0:
                selected = version
                runtime_count = count
                break

        except Exception as e:
            last_error = str(e)[-500:]

    retry_results.append({
        "contract_id": contract_id,
        "compiler_found": selected,
        "runtime_contracts": runtime_count,
        "status": "SUCCESS" if selected else "FAILED",
        "error": "" if selected else last_error
    })

df_retry = pd.DataFrame(retry_results)

print("========== KẾT QUẢ THỬ LẠI ==========")
print(
    df_retry[
        ["contract_id", "compiler_found", "runtime_contracts", "status"]
    ].to_string(index=False)
)

print("\n========== THỐNG KÊ ==========")
print(df_retry["status"].value_counts())

print("\nCác file vẫn thất bại:")
print(
    df_retry[df_retry["status"] == "FAILED"]
    .to_string(index=False)
)

========== KẾT QUẢ THỬ LẠI ==========
contract_id compiler_found  runtime_contracts  status
    C000010         0.5.17                  4 SUCCESS
    C000015          0.5.2                  4 SUCCESS
    C000022         0.5.17                  2 SUCCESS
    C000025         0.5.17                  2 SUCCESS
    C000027         0.5.17                  3 SUCCESS
    C000028         0.5.17                  3 SUCCESS
    C000029         0.5.17                  3 SUCCESS
    C000030          0.5.2                  4 SUCCESS
    C000060         0.5.17                  4 SUCCESS
    C000065          0.5.2                  4 SUCCESS
    C000072         0.5.17                  2 SUCCESS
    C000075         0.5.17                  2 SUCCESS
    C000077         0.5.17                  3 SUCCESS
    C000078         0.5.17                  3 SUCCESS
    C000079         0.5.17                  3 SUCCESS
    C000080          0.5.2                  4 SUCCESS

========== THỐNG KÊ ==========
status
SUCCE

In [55]:
# Tạo bản mapping mới, giữ nguyên mapping ban đầu
df_compiler_mapping_v2 = df_compiler_mapping.copy()

# Cập nhật compiler cho 16 file đã thử lại thành công
for _, row in df_retry.iterrows():

    if row["status"] != "SUCCESS":
        continue

    mask = (
        df_compiler_mapping_v2["contract_id"]
        == row["contract_id"]
    )

    df_compiler_mapping_v2.loc[
        mask, "selected_compiler"
    ] = row["compiler_found"]

print("========== MAPPING ĐÃ CẬP NHẬT ==========")

print(
    df_compiler_mapping_v2[
        df_compiler_mapping_v2["contract_id"].isin(
            df_retry["contract_id"]
        )
    ][
        ["contract_id", "selected_compiler"]
    ].to_string(index=False)
)

print("\nTổng số file:", len(df_compiler_mapping_v2))

========== MAPPING ĐÃ CẬP NHẬT ==========
contract_id selected_compiler
    C000010            0.5.17
    C000015             0.5.2
    C000022            0.5.17
    C000025            0.5.17
    C000027            0.5.17
    C000028            0.5.17
    C000029            0.5.17
    C000030             0.5.2
    C000060            0.5.17
    C000065             0.5.2
    C000072            0.5.17
    C000075            0.5.17
    C000077            0.5.17
    C000078            0.5.17
    C000079            0.5.17
    C000080             0.5.2

Tổng số file: 39905


In [57]:
installed_versions = sorted(
    [str(v) for v in solcx.get_installed_solc_versions()],
    key=semantic_version.Version
)

# Mapping
mapping_lookup = df_compiler_mapping_v2.set_index(
    "contract_id"
).to_dict("index")


def get_compatible_versions(source):
    pragmas = re.findall(
        r"pragma\s+solidity\s+([^;]+);",
        source
    )

    if not pragmas:
        return installed_versions.copy()

    compatible = []

    for version in installed_versions:
        try:
            v = semantic_version.Version(version)

            if all(
                semantic_version.NpmSpec(p.strip()).match(v)
                for p in pragmas
            ):
                compatible.append(version)

        except (ValueError, TypeError):
            continue

    return compatible


def try_compile_file(contract_id):

    file = file_lookup[contract_id]
    mapping = mapping_lookup[contract_id]

    # Compiler lịch sử: xử lý riêng bằng solc-js
    if mapping["status"] == "NO_COMPATIBLE_COMPILER":
        return {
            "contract_id": contract_id,
            "status": "LEGACY_PENDING",
            "compiler": None,
            "attempts": 0,
            "runtime_contracts": 0,
            "error": "Cần kiểm tra bằng solc-js"
        }

    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    compatible = get_compatible_versions(source)

    preferred = mapping.get("selected_compiler")

    # Không có pragma: ưu tiên 0.4.26
    if mapping["status"] == "NO_PRAGMA":
        preferred = "0.4.26"

    # Ưu tiên compiler trong mapping
    candidates = []

    if preferred in compatible:
        candidates.append(preferred)

    # Thử compiler khác từ cũ đến mới
    candidates.extend(
        v for v in compatible
        if v not in candidates
    )

    if not candidates:
        return {
            "contract_id": contract_id,
            "status": "NO_COMPILER",
            "compiler": None,
            "attempts": 0,
            "runtime_contracts": 0,
            "error": "Không có compiler tương thích trong danh sách"
        }

    attempts = 0
    last_error = ""

    for compiler in candidates:
        attempts += 1

        try:
            compiled = solcx.compile_source(
                source,
                output_values=["bin-runtime"],
                solc_version=compiler
            )

            runtime_count = sum(
                bool(item.get("bin-runtime"))
                for item in compiled.values()
            )

            if runtime_count > 0:
                return {
                    "contract_id": contract_id,
                    "status": "SUCCESS",
                    "compiler": compiler,
                    "attempts": attempts,
                    "runtime_contracts": runtime_count,
                    "error": ""
                }

            last_error = "Không có Runtime Bytecode"

        except Exception as e:
            error_text = str(e)

            if "> stderr:" in error_text:
                error_text = error_text.split(
                    "> stderr:", 1
                )[1]

            last_error = error_text[:2000]

    return {
        "contract_id": contract_id,
        "status": "FAILED",
        "compiler": None,
        "attempts": attempts,
        "runtime_contracts": 0,
        "error": last_error
    }

all_results = []
start_time = time.time()

contract_ids = df_compiler_mapping_v2[
    "contract_id"
].tolist()

total = len(contract_ids)

for i, contract_id in enumerate(contract_ids, start=1):

    result = try_compile_file(contract_id)
    all_results.append(result)

    if i % 1000 == 0 or i == total:
        elapsed = time.time() - start_time

        counts = Counter(
            item["status"] for item in all_results
        )

        print(
            f"[{i:,}/{total:,}] "
            f"SUCCESS={counts['SUCCESS']:,} | "
            f"FAILED={counts['FAILED']:,} | "
            f"NO_COMPILER={counts['NO_COMPILER']:,} | "
            f"LEGACY_PENDING={counts['LEGACY_PENDING']:,} | "
            f"Thời gian={elapsed/60:.1f} phút",
            flush=True
        )


# Kết quả chỉ nằm trong RAM Colab
df_compile_all = pd.DataFrame(all_results)

print("\n========== KẾT QUẢ CUỐI CÙNG ==========")
print(df_compile_all["status"].value_counts())

print("\n========== COMPILER THÀNH CÔNG ==========")
print(
    df_compile_all.loc[
        df_compile_all["status"] == "SUCCESS",
        "compiler"
    ].value_counts()
)

print("\n========== SỐ LẦN THỬ ==========")
print(df_compile_all["attempts"].describe())

print(
    f"\nTổng thời gian: "
    f"{(time.time() - start_time)/60:.2f} phút"
)

[1,000/39,905] SUCCESS=969 | FAILED=0 | NO_COMPILER=29 | LEGACY_PENDING=2 | Thời gian=0.7 phút
[2,000/39,905] SUCCESS=1,953 | FAILED=0 | NO_COMPILER=45 | LEGACY_PENDING=2 | Thời gian=1.6 phút
[3,000/39,905] SUCCESS=2,935 | FAILED=0 | NO_COMPILER=61 | LEGACY_PENDING=4 | Thời gian=2.4 phút
[4,000/39,905] SUCCESS=3,918 | FAILED=0 | NO_COMPILER=77 | LEGACY_PENDING=5 | Thời gian=3.2 phút
[5,000/39,905] SUCCESS=4,907 | FAILED=0 | NO_COMPILER=87 | LEGACY_PENDING=6 | Thời gian=4.0 phút
[6,000/39,905] SUCCESS=5,892 | FAILED=0 | NO_COMPILER=99 | LEGACY_PENDING=9 | Thời gian=4.8 phút
[7,000/39,905] SUCCESS=6,873 | FAILED=0 | NO_COMPILER=116 | LEGACY_PENDING=11 | Thời gian=5.6 phút
[8,000/39,905] SUCCESS=7,863 | FAILED=0 | NO_COMPILER=124 | LEGACY_PENDING=13 | Thời gian=6.4 phút
[9,000/39,905] SUCCESS=8,850 | FAILED=0 | NO_COMPILER=136 | LEGACY_PENDING=14 | Thời gian=7.2 phút
[10,000/39,905] SUCCESS=9,845 | FAILED=0 | NO_COMPILER=140 | LEGACY_PENDING=15 | Thời gian=8.1 phút
[11,000/39,905] SUCCESS

In [58]:
no_compiler_ids = df_compile_all.loc[
    df_compile_all["status"] == "NO_COMPILER",
    "contract_id"
].tolist()

pragma_counter = Counter()
examples = {}

for contract_id in no_compiler_ids:
    source = file_lookup[contract_id].read_text(
        encoding="utf-8",
        errors="replace"
    )

    pragmas = re.findall(
        r"pragma\s+solidity\s+([^;]+);",
        source
    )

    key = " | ".join(pragmas) if pragmas else "NO_PRAGMA"

    pragma_counter[key] += 1
    examples.setdefault(key, contract_id)

print("========== THỐNG KÊ 520 FILE ==========")
print("Tổng số file:", len(no_compiler_ids))
print("Số kiểu pragma:", len(pragma_counter))

print("\nCác pragma phổ biến:")
for pragma, count in pragma_counter.most_common(30):
    print(
        f"{count:>5} file | "
        f"{pragma:<35} | "
        f"Ví dụ: {examples[pragma]}"
    )

========== THỐNG KÊ 520 FILE ==========
Tổng số file: 520
Số kiểu pragma: 101

Các pragma phổ biến:
  127 file | ^ 0.4 .9                            | Ví dụ: C000868
   26 file | ^ 0.4.24                            | Ví dụ: C002043
   22 file | ^ 0.4.18                            | Ví dụ: C001928
   19 file | ^ 0.4.17                            | Ví dụ: C002116
   11 file | ^0.5.00                             | Ví dụ: C000031
   11 file | ^ 0.4 .2                            | Ví dụ: C000555
   10 file | ^ 0.4 .24                           | Ví dụ: C001102
   10 file | ^ 0.4 .11                           | Ví dụ: C001731
   10 file | ^ 0.4.19                            | Ví dụ: C003177
   10 file | ^ 0.4.25                            | Ví dụ: C008126
   10 file | ^ 0.4.21                            | Ví dụ: C011547
    9 file | > 0.4.99 <0.6.0                     | Ví dụ: C003928
    9 file | ^ 0.4.16                            | Ví dụ: C007365
    8 file | ^ 0.4 .19                    

In [59]:
# Loại bỏ comment để không đọc nhầm pragma trong comment.
# Chỉ dùng để phân tích pragma, KHÔNG dùng để biên dịch.
def remove_comments_for_pragma(source):
    pattern = r'//[^\n]*|/\*[\s\S]*?\*/'

    return re.sub(pattern, "", source)


def normalize_pragma(pragma):
    pragma = pragma.strip()

    # Chuẩn hóa khoảng trắng trong số phiên bản
    pragma = re.sub(
        r'(\d+)\s*\.\s*(\d+)\s*\.\s*(\d+)',
        r'\1.\2.\3',
        pragma
    )

    # Chuẩn hóa 0.5.00 -> 0.5.0
    pragma = re.sub(
        r'\d+\.\d+\.\d+',
        lambda m: ".".join(
            str(int(x)) for x in m.group().split(".")
        ),
        pragma
    )

    # Chuẩn hóa khoảng trắng giữa toán tử và phiên bản
    pragma = re.sub(
        r'(\^|~|>=|<=|>|<|=)\s+',
        r'\1',
        pragma
    )

    return pragma


def get_normalized_pragmas(source):
    cleaned_source = remove_comments_for_pragma(source)

    pragmas = re.findall(
        r'\bpragma\s+solidity\s+([^;]+);',
        cleaned_source
    )

    return [
        normalize_pragma(p)
        for p in pragmas
    ]


def find_compatible_compilers(source):
    pragmas = get_normalized_pragmas(source)

    compatible = []

    for version in installed_versions:
        try:
            v = semantic_version.Version(version)

            if all(
                semantic_version.NpmSpec(p).match(v)
                for p in pragmas
            ):
                compatible.append(version)

        except (ValueError, TypeError):
            continue

    return compatible


# Kiểm tra lại 520 file
results = []

for contract_id in no_compiler_ids:

    source = file_lookup[contract_id].read_text(
        encoding="utf-8",
        errors="replace"
    )

    pragmas = get_normalized_pragmas(source)
    compatible = find_compatible_compilers(source)

    results.append({
        "contract_id": contract_id,
        "normalized_pragma": " | ".join(pragmas),
        "compatible_count": len(compatible),
        "compatible_compilers": compatible
    })

df_pragma_check = pd.DataFrame(results)

print("========== KẾT QUẢ ==========")

print("Tổng số file:", len(df_pragma_check))

print(
    "Đã tìm được compiler:",
    (df_pragma_check["compatible_count"] > 0).sum()
)

print(
    "Vẫn chưa có compiler:",
    (df_pragma_check["compatible_count"] == 0).sum()
)

print("\n========== 20 FILE ĐẦU ==========")

print(
    df_pragma_check[
        [
            "contract_id",
            "normalized_pragma",
            "compatible_compilers"
        ]
    ].head(20).to_string(index=False)
)

print("\n========== FILE CHƯA CÓ COMPILER ==========")

print(
    df_pragma_check.loc[
        df_pragma_check["compatible_count"] == 0,
        ["contract_id", "normalized_pragma"]
    ].head(30).to_string(index=False)
)

========== KẾT QUẢ ==========
Tổng số file: 520
Đã tìm được compiler: 511
Vẫn chưa có compiler: 9

========== 20 FILE ĐẦU ==========
contract_id normalized_pragma                                                        compatible_compilers
    C000002   >=0.4.21 <0.6.0 [0.4.21, 0.4.24, 0.4.25, 0.4.26, 0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000031            ^0.5.0                                 [0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000034    >=0.5.0 <0.6.0                                 [0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000052   >=0.4.21 <0.6.0 [0.4.21, 0.4.24, 0.4.25, 0.4.26, 0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000081            ^0.5.0                                 [0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000084    >=0.5.0 <0.6.0                                 [0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000102   >=0.4.21 <0.6.0 [0.4.21, 0.4.24, 0.4.25, 0.4.26, 0.5.0, 0.5.2, 0.5.6, 0.5.7, 0.5.8, 0.5.17]
    C000131        

In [60]:
def normalize_pragma_v2(pragma):
    pragma = normalize_pragma(pragma)

    # Bỏ dấu ngoặc kép khi phân tích phiên bản
    pragma = pragma.replace('"', "").replace("'", "")

    # Tách các điều kiện viết liền nhau
    pragma = re.sub(
        r'(\d)(?=(?:>=|<=|>|<|=|\^|~))',
        r'\1 ',
        pragma
    )

    return pragma.strip()


def get_candidate_versions_v2(source):
    cleaned = remove_comments_for_pragma(source)

    pragmas = re.findall(
        r'\bpragma\s+solidity\s+([^;]+);',
        cleaned
    )

    pragmas = [
        normalize_pragma_v2(p)
        for p in pragmas
    ]

    compatible = []

    for version in installed_versions:
        try:
            v = semantic_version.Version(version)

            if all(
                semantic_version.NpmSpec(p).match(v)
                for p in pragmas
            ):
                compatible.append(version)

        except (ValueError, TypeError):
            continue

    return compatible


results_520 = []
start = time.time()

for i, contract_id in enumerate(no_compiler_ids, 1):

    file = file_lookup[contract_id]
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    candidates = get_candidate_versions_v2(source)

    selected = None
    runtime_count = 0
    attempts = 0
    last_error = ""

    for compiler in candidates:
        attempts += 1

        try:
            compiled = solcx.compile_source(
                source,
                output_values=["bin-runtime"],
                solc_version=compiler
            )

            count = sum(
                bool(item.get("bin-runtime"))
                for item in compiled.values()
            )

            if count > 0:
                selected = compiler
                runtime_count = count
                break

            last_error = "Không có Runtime Bytecode"

        except Exception as e:
            last_error = str(e)[-1000:]

    if selected:
        status = "SUCCESS"
    elif not candidates:
        status = "NO_COMPILER"
    else:
        status = "FAILED"

    results_520.append({
        "contract_id": contract_id,
        "status": status,
        "compiler": selected,
        "attempts": attempts,
        "runtime_contracts": runtime_count,
        "error": "" if selected else last_error
    })

    if i % 100 == 0:
        counts = Counter(
            r["status"] for r in results_520
        )
        print(
            f"[{i}/{len(no_compiler_ids)}] "
            f"SUCCESS={counts['SUCCESS']} | "
            f"FAILED={counts['FAILED']} | "
            f"NO_COMPILER={counts['NO_COMPILER']}",
            flush=True
        )

df_retry_520 = pd.DataFrame(results_520)

print("\n========== KẾT QUẢ 520 FILE ==========")
print(df_retry_520["status"].value_counts())

print("\n========== COMPILER THÀNH CÔNG ==========")
print(
    df_retry_520.loc[
        df_retry_520["status"] == "SUCCESS",
        "compiler"
    ].value_counts()
)

print("\n========== FILE CHƯA THÀNH CÔNG ==========")
print(
    df_retry_520.loc[
        df_retry_520["status"] != "SUCCESS",
        ["contract_id", "status", "attempts", "error"]
    ].to_string(index=False, max_colwidth=150)
)

print(
    f"\nThời gian: {(time.time()-start)/60:.2f} phút"
)

[100/520] SUCCESS=100 | FAILED=0 | NO_COMPILER=0
[200/520] SUCCESS=200 | FAILED=0 | NO_COMPILER=0
[300/520] SUCCESS=300 | FAILED=0 | NO_COMPILER=0
[400/520] SUCCESS=400 | FAILED=0 | NO_COMPILER=0
[500/520] SUCCESS=500 | FAILED=0 | NO_COMPILER=0

========== KẾT QUẢ 520 FILE ==========
status
SUCCESS    520
Name: count, dtype: int64

========== COMPILER THÀNH CÔNG ==========
compiler
0.4.11    183
0.4.24    124
0.5.0      52
0.4.17     35
0.4.21     26
0.4.25     25
0.4.18     24
0.4.19     22
0.5.6      13
0.5.7       4
0.4.14      4
0.5.8       4
0.5.2       3
0.4.26      1
Name: count, dtype: int64

========== FILE CHƯA THÀNH CÔNG ==========
Empty DataFrame
Columns: [contract_id, status, attempts, error]
Index: []

Thời gian: 0.56 phút


In [61]:
legacy_versions = {
    "0.4.4": "v0.4.4+commit.4633f3de",
    "0.4.6": "v0.4.6+commit.2dabbdf0",
    "0.4.8": "v0.4.8+commit.60cc1668",
    "0.4.9": "v0.4.9+commit.364da425",
    "0.4.10": "v0.4.10+commit.f0d539ae"
}

legacy_ids = df_compile_all.loc[
    df_compile_all["status"] == "LEGACY_PENDING",
    "contract_id"
].tolist()

results_legacy = []
start = time.time()

for i, contract_id in enumerate(legacy_ids, 1):
    file = file_lookup[contract_id]
    source = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    pragmas = get_normalized_pragmas(source)

    candidates = []

    for version in legacy_versions:
        try:
            v = semantic_version.Version(version)
            if all(
                semantic_version.NpmSpec(p).match(v)
                for p in pragmas
            ):
                candidates.append(version)
        except (ValueError, TypeError):
            continue

    selected = None
    runtime_count = 0
    last_error = ""
    attempts = 0

    for version in candidates:
        attempts += 1

        try:
            result = subprocess.run(
                [
                    "node",
                    str(script_path),
                    str(file),
                    legacy_versions[version]
                ],
                cwd=str(LEGACY_DIR),
                capture_output=True,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                last_error = result.stderr[-1000:]
                continue

            output = json.loads(result.stdout)

            if output.get("errors"):
                last_error = str(output["errors"])[-1000:]
                continue

            runtimes = output.get("runtimes", {})

            count = sum(
                bool(bytecode)
                for bytecode in runtimes.values()
            )

            if count > 0:
                selected = version
                runtime_count = count
                break

            last_error = "Không có Runtime Bytecode"

        except Exception as e:
            last_error = str(e)[-1000:]

    status = "SUCCESS" if selected else "FAILED"

    results_legacy.append({
        "contract_id": contract_id,
        "status": status,
        "compiler": selected,
        "attempts": attempts,
        "runtime_contracts": runtime_count,
        "error": "" if selected else last_error
    })

    if i % 10 == 0 or i == len(legacy_ids):
        counts = Counter(
            r["status"] for r in results_legacy
        )

        print(
            f"[{i}/{len(legacy_ids)}] "
            f"SUCCESS={counts['SUCCESS']} | "
            f"FAILED={counts['FAILED']}",
            flush=True
        )

df_legacy = pd.DataFrame(results_legacy)

print("\n========== KẾT QUẢ 53 FILE ==========")
print(df_legacy["status"].value_counts())

print("\n========== COMPILER THÀNH CÔNG ==========")
print(
    df_legacy.loc[
        df_legacy["status"] == "SUCCESS",
        "compiler"
    ].value_counts()
)

print("\n========== FILE THẤT BẠI ==========")
print(
    df_legacy.loc[
        df_legacy["status"] != "SUCCESS",
        ["contract_id", "attempts", "error"]
    ].to_string(index=False, max_colwidth=150)
)

print(
    f"\nThời gian: {(time.time() - start)/60:.2f} phút"
)

[10/53] SUCCESS=10 | FAILED=0
[20/53] SUCCESS=20 | FAILED=0
[30/53] SUCCESS=30 | FAILED=0
[40/53] SUCCESS=40 | FAILED=0
[50/53] SUCCESS=50 | FAILED=0
[53/53] SUCCESS=53 | FAILED=0

========== KẾT QUẢ 53 FILE ==========
status
SUCCESS    53
Name: count, dtype: int64

========== COMPILER THÀNH CÔNG ==========
compiler
0.4.8     27
0.4.4     12
0.4.10    10
0.4.9      2
0.4.6      2
Name: count, dtype: int64

========== FILE THẤT BẠI ==========
Empty DataFrame
Columns: [contract_id, attempts, error]
Index: []

Thời gian: 3.39 phút


In [62]:

# 1. Kết quả biên dịch ban đầu
df_final_compiler = df_compile_all.copy()

# 2. Cập nhật 520 file đã khắc phục
retry_lookup = df_retry_520.set_index("contract_id")

for contract_id in retry_lookup.index:
    mask = df_final_compiler["contract_id"] == contract_id

    df_final_compiler.loc[mask, "status"] = (
        retry_lookup.loc[contract_id, "status"]
    )
    df_final_compiler.loc[mask, "compiler"] = (
        retry_lookup.loc[contract_id, "compiler"]
    )
    df_final_compiler.loc[mask, "runtime_contracts"] = (
        retry_lookup.loc[contract_id, "runtime_contracts"]
    )

# 3. Cập nhật 53 file compiler lịch sử
legacy_lookup = df_legacy.set_index("contract_id")

for contract_id in legacy_lookup.index:
    mask = df_final_compiler["contract_id"] == contract_id

    df_final_compiler.loc[mask, "status"] = (
        legacy_lookup.loc[contract_id, "status"]
    )
    df_final_compiler.loc[mask, "compiler"] = (
        legacy_lookup.loc[contract_id, "compiler"]
    )
    df_final_compiler.loc[mask, "runtime_contracts"] = (
        legacy_lookup.loc[contract_id, "runtime_contracts"]
    )

# 4. Kiểm tra tính toàn vẹn
print("========== MAPPING CUỐI CÙNG ==========")
print("Tổng số file:", len(df_final_compiler))
print("Contract ID duy nhất:", df_final_compiler["contract_id"].nunique())

print("\nTrạng thái:")
print(df_final_compiler["status"].value_counts())

print("\nCompiler được sử dụng:")
print(df_final_compiler["compiler"].value_counts())

assert len(df_final_compiler) == 39905
assert df_final_compiler["contract_id"].nunique() == 39905
assert (df_final_compiler["status"] == "SUCCESS").all()
assert df_final_compiler["compiler"].notna().all()

print("\nPASS: Mapping compiler đầy đủ và hợp lệ!")

========== MAPPING CUỐI CÙNG ==========
Tổng số file: 39905
Contract ID duy nhất: 39905

Trạng thái:
status
SUCCESS    39905
Name: count, dtype: int64

Compiler được sử dụng:
compiler
0.4.26    33646
0.5.17     2584
0.4.24     1399
0.4.25      619
0.4.11      444
0.4.19      337
0.4.21      263
0.5.0       136
0.5.2       101
0.5.7        90
0.5.8        66
0.5.6        57
0.4.17       52
0.4.18       34
0.4.8        27
0.4.14       16
0.4.4        12
0.4.10       10
0.7.6         6
0.4.9         2
0.4.6         2
0.6.12        2
Name: count, dtype: int64

PASS: Mapping compiler đầy đủ và hợp lệ!


# **TRÍCH XUẤT OPCODE**

In [63]:
opcode_results = []
opcode_start = time.time()

total = len(df_final_compiler)

for i, row in enumerate(
    df_final_compiler.itertuples(index=False),
    start=1
):
    contract_id = row.contract_id
    compiler = str(row.compiler)
    file = file_lookup[contract_id]

    try:
        # 1. Biên dịch và lấy Runtime Bytecode
        if compiler in legacy_versions:
            result = subprocess.run(
                [
                    "node",
                    str(script_path),
                    str(file),
                    legacy_versions[compiler]
                ],
                cwd=str(LEGACY_DIR),
                capture_output=True,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                raise RuntimeError(result.stderr[-1000:])

            output = json.loads(result.stdout)

            if output.get("errors"):
                raise RuntimeError(str(output["errors"])[-1000:])

            runtimes = list(
                output.get("runtimes", {}).values()
            )

        else:
            source = file.read_text(
                encoding="utf-8",
                errors="replace"
            )

            compiled = solcx.compile_source(
                source,
                output_values=["bin-runtime"],
                solc_version=compiler
            )

            runtimes = [
                artifact.get("bin-runtime", "")
                for artifact in compiled.values()
            ]

        # 2. Trích xuất Opcode
        counts = Counter()
        runtime_contracts = 0
        metadata_removed = 0
        metadata_bytes_removed = 0

        for runtime in runtimes:
            if not runtime:
                continue

            # Không âm thầm bỏ qua bytecode chưa liên kết thư viện
            if not all(
                c in "0123456789abcdefABCDEF"
                for c in runtime.removeprefix("0x")
            ):
                raise ValueError(
                    "Runtime Bytecode chứa ký tự không phải hex"
                )

            clean, removed, removed_bytes = (
                strip_solidity_metadata(runtime)
            )

            opcode_counts = extract_opcode_counts(clean)

            counts.update(opcode_counts)
            runtime_contracts += 1

            if removed:
                metadata_removed += 1
                metadata_bytes_removed += removed_bytes

        if runtime_contracts == 0:
            raise ValueError("Không có Runtime Bytecode")

        opcode_results.append({
            "contract_id": contract_id,
            "compiler": compiler,
            "status": "SUCCESS",
            "runtime_contracts": runtime_contracts,
            "metadata_removed": metadata_removed,
            "metadata_bytes_removed": metadata_bytes_removed,
            "total_opcodes": sum(counts.values()),
            "opcode_counts": dict(counts),
            "error": ""
        })

    except Exception as e:
        opcode_results.append({
            "contract_id": contract_id,
            "compiler": compiler,
            "status": "FAILED",
            "runtime_contracts": 0,
            "metadata_removed": 0,
            "metadata_bytes_removed": 0,
            "total_opcodes": 0,
            "opcode_counts": {},
            "error": str(e)[-1000:]
        })

    # In tiến độ
    if i % 1000 == 0 or i == total:
        summary = Counter(
            r["status"] for r in opcode_results
        )

        elapsed = (time.time() - opcode_start) / 60

        print(
            f"[{i:,}/{total:,}] "
            f"SUCCESS={summary['SUCCESS']:,} | "
            f"FAILED={summary['FAILED']:,} | "
            f"Thời gian={elapsed:.1f} phút",
            flush=True
        )


# 3. Chuyển kết quả thành DataFrame trong RAM
df_opcode_results = pd.DataFrame(opcode_results)

print("\n========== KẾT QUẢ TRÍCH XUẤT OPCODE ==========")
print(df_opcode_results["status"].value_counts())

print("\n========== THỐNG KÊ ==========")

success = df_opcode_results[
    df_opcode_results["status"] == "SUCCESS"
]

print("Số file thành công:", len(success))
print("Tổng Runtime Contract:", success["runtime_contracts"].sum())
print("Số Runtime Contract đã loại metadata:", success["metadata_removed"].sum())
print("Tổng Opcode:", success["total_opcodes"].sum())

print("\nPhân bố số Opcode trên mỗi file:")
print(success["total_opcodes"].describe())

print("\n========== FILE LỖI ==========")
print(
    df_opcode_results.loc[
        df_opcode_results["status"] == "FAILED",
        ["contract_id", "compiler", "error"]
    ].head(30).to_string(index=False, max_colwidth=120)
)

print(
    f"\nTổng thời gian: "
    f"{(time.time() - opcode_start)/60:.2f} phút"
)

[1,000/39,905] SUCCESS=1,000 | FAILED=0 | Thời gian=1.0 phút
[2,000/39,905] SUCCESS=2,000 | FAILED=0 | Thời gian=2.0 phút
[3,000/39,905] SUCCESS=3,000 | FAILED=0 | Thời gian=3.0 phút
[4,000/39,905] SUCCESS=4,000 | FAILED=0 | Thời gian=4.0 phút
[5,000/39,905] SUCCESS=5,000 | FAILED=0 | Thời gian=5.0 phút
[6,000/39,905] SUCCESS=6,000 | FAILED=0 | Thời gian=6.1 phút
[7,000/39,905] SUCCESS=7,000 | FAILED=0 | Thời gian=7.2 phút
[8,000/39,905] SUCCESS=8,000 | FAILED=0 | Thời gian=8.2 phút
[9,000/39,905] SUCCESS=9,000 | FAILED=0 | Thời gian=9.2 phút
[10,000/39,905] SUCCESS=10,000 | FAILED=0 | Thời gian=10.2 phút
[11,000/39,905] SUCCESS=11,000 | FAILED=0 | Thời gian=11.1 phút
[12,000/39,905] SUCCESS=12,000 | FAILED=0 | Thời gian=12.0 phút
[13,000/39,905] SUCCESS=13,000 | FAILED=0 | Thời gian=13.1 phút
[14,000/39,905] SUCCESS=14,000 | FAILED=0 | Thời gian=14.0 phút
[15,000/39,905] SUCCESS=15,000 | FAILED=0 | Thời gian=15.0 phút
[16,000/39,905] SUCCESS=16,000 | FAILED=0 | Thời gian=16.1 phút
[17

**Kiểm tra chất lượng Opcode & Bảng đặc trưng**

In [69]:


label_path = "/content/drive/MyDrive/IS6002/EXPERIMENT/implementation/data/contracts_and_label.csv"

df_labels = pd.read_csv(label_path)

print("========== THÔNG TIN FILE NHÃN ==========")
print("Số dòng:", len(df_labels))

print("\nDanh sách cột:")
print(df_labels.columns.tolist())

print("\n5 dòng đầu tiên:")
print(df_labels.head(5).to_string(index=False))

print("\n========== KIỂM TRA OPCODE ==========")
print("Số file Opcode:", len(df_opcode_results))
print("Contract ID duy nhất:",
      df_opcode_results["contract_id"].nunique())

print("\nVí dụ Opcode:")
print(
    df_opcode_results[
        ["contract_id", "total_opcodes", "opcode_counts"]
    ].head(3).to_string(index=False, max_colwidth=120)
)

========== THÔNG TIN FILE NHÃN ==========
Số dòng: 39905

Danh sách cột:
['contract_id', 'label']

5 dòng đầu tiên:
contract_id      label
    C000001 Vulnerable
    C000002 Vulnerable
    C000003 Vulnerable
    C000004 Vulnerable
    C000005 Vulnerable

========== KIỂM TRA OPCODE ==========
Số file Opcode: 39905
Contract ID duy nhất: 39905

Ví dụ Opcode:
contract_id  total_opcodes                                                                                                            opcode_counts
    C000001           3331 {'PUSH1': 481, 'OP_52': 109, 'OP_34': 1, 'OP_80': 247, 'OP_15': 75, 'PUSH2': 173, 'OP_57': 85, 'OP_FD': 33, 'OP_5B': ...
    C000002           2617 {'PUSH1': 366, 'OP_52': 83, 'OP_36': 13, 'OP_10': 12, 'PUSH2': 142, 'OP_57': 67, 'OP_35': 16, 'PUSH29': 1, 'OP_90': 1...
    C000003           7851 {'PUSH1': 1081, 'OP_52': 218, 'OP_36': 45, 'OP_10': 62, 'PUSH2': 455, 'OP_57': 243, 'OP_35': 64, 'OP_1C': 2, 'OP_80':...


In [77]:
!pip -q install pyevmasm

from pyevmasm import instruction_tables

assert len(df_opcode_results) == 39905
assert df_opcode_results["contract_id"].is_unique
assert (df_opcode_results["status"] == "SUCCESS").all()

assert len(df_labels) == 39905
assert df_labels["contract_id"].is_unique



df_opcode_matrix = pd.DataFrame(
    df_opcode_results["opcode_counts"].tolist(),
    index=df_opcode_results["contract_id"]
)

df_opcode_matrix = (
    df_opcode_matrix
    .fillna(0)
    .astype("int64")
)

df_opcode_matrix.index.name = "contract_id"

opcode_table = instruction_tables["istanbul"]

rename_map = {}

for col in df_opcode_matrix.columns:

    match = re.fullmatch(r"OP_([0-9A-Fa-f]{2})", col)

    if match:
        opcode_byte = int(match.group(1), 16)
        instruction = opcode_table.get(opcode_byte)

        if instruction is not None:
            rename_map[col] = instruction.name

new_columns = [
    rename_map.get(col, col)
    for col in df_opcode_matrix.columns
]

assert len(new_columns) == len(set(new_columns)), (
    "Phát hiện tên Opcode bị trùng"
)

df_opcode_matrix = df_opcode_matrix.rename(
    columns=rename_map
)

df_opcode_matrix = df_opcode_matrix.reindex(
    sorted(df_opcode_matrix.columns),
    axis=1
)

df_opcode_matrix = df_opcode_matrix.reset_index()


df_labels_final = df_labels[
    ["contract_id", "label"]
].copy()

df_final_dataset = df_opcode_matrix.merge(
    df_labels_final,
    on="contract_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(df_final_dataset) == 39905
assert df_final_dataset["contract_id"].is_unique
assert (df_final_dataset["_merge"] == "both").all()

df_final_dataset = df_final_dataset.drop(columns=["_merge"])

opcode_columns = [
    c for c in df_final_dataset.columns
    if c not in ["contract_id", "label"]
]

original_totals = df_opcode_results.set_index(
    "contract_id"
)["total_opcodes"]

computed_totals = df_final_dataset.set_index(
    "contract_id"
)[opcode_columns].sum(axis=1)

assert computed_totals.equals(
    original_totals.reindex(computed_totals.index)
), "Tổng Opcode không khớp!"

output_dir = Path(
    "/content/drive/MyDrive/IS6002/EXPERIMENT/implementation/data"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "opcode_by_contract.csv"

df_final_dataset.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("========== TẠO CSV THÀNH CÔNG ==========")

print("Đường dẫn:", output_path)
print("Số dòng:", len(df_final_dataset))
print("Số cột Opcode:", len(opcode_columns))
print("Tổng số cột:", len(df_final_dataset.columns))

print("\nSố Opcode được đổi tên:", len(rename_map))

print("\nVí dụ ánh xạ:")
for old, new in list(rename_map.items())[:20]:
    print(f"{old:10} -> {new}")

print("\nCác Opcode chưa nhận diện:")
print([
    col for col in opcode_columns
    if re.fullmatch(r"OP_[0-9A-Fa-f]{2}", col)
])

print("\nPhân bố label:")
print(df_final_dataset["label"].value_counts(dropna=False))

print("\n5 dòng đầu:")
print(
    df_final_dataset[
        ["contract_id"] + opcode_columns[:8] + ["label"]
    ].head().to_string(index=False)
)

print(
    "\nDung lượng CSV:",
    round(output_path.stat().st_size / 1024**2, 2),
    "MB"
)

========== TẠO CSV THÀNH CÔNG ==========
Đường dẫn: /content/drive/MyDrive/IS6002/EXPERIMENT/implementation/data/opcode_by_contract.csv
Số dòng: 39905
Số cột Opcode: 256
Tổng số cột: 258

Số Opcode được đổi tên: 110

Ví dụ ánh xạ:
OP_52      -> MSTORE
OP_34      -> CALLVALUE
OP_80      -> DUP1
OP_15      -> ISZERO
OP_57      -> JUMPI
OP_FD      -> REVERT
OP_5B      -> JUMPDEST
OP_50      -> POP
OP_36      -> CALLDATASIZE
OP_10      -> LT
OP_35      -> CALLDATALOAD
OP_1C      -> SHR
OP_11      -> GT
OP_14      -> EQ
OP_56      -> JUMP
OP_51      -> MLOAD
OP_01      -> ADD
OP_82      -> DUP3
OP_81      -> DUP2
OP_03      -> SUB

Các Opcode chưa nhận diện:
['OP_0C', 'OP_0D', 'OP_0E', 'OP_0F', 'OP_1E', 'OP_1F', 'OP_21', 'OP_22', 'OP_23', 'OP_24', 'OP_25', 'OP_26', 'OP_27', 'OP_28', 'OP_29', 'OP_2A', 'OP_2B', 'OP_2C', 'OP_2D', 'OP_2E', 'OP_2F', 'OP_48', 'OP_49', 'OP_4A', 'OP_4B', 'OP_4C', 'OP_4D', 'OP_4E', 'OP_4F', 'OP_5C', 'OP_5D', 'OP_5E', 'OP_5F', 'OP_A5', 'OP_A6', 'OP_A7', 'OP_A8', 'OP_